In [ ]:
# ============================================================
# 01-period-recovery.ipynb
# ============================================================

In [ ]:
from astropy import table

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import (
    GaiaData,
    plot_raw_phase_folded_lightcurve,
    attach_flux_mean_magnitudes,
    attach_periodogram_periods,
    plot_lomb_scargle_periodogram,
    plot_vari_rrlyrae_period_comparison,
    plot_fourier_harmonic_fits,
    cross_validate_harmonics,
    fourier_fit,
    fourier_mean_magnitude,
    fourier_mean_magnitude_error,
    plot_fourier_cross_validation,
    plot_fourier_cv_normalized_residual_histograms,
    plot_fourier_cv_phase_comparison,
    plot_fourier_extrapolation,
    plot_mean_g_catalog_comparison,
)

In [ ]:
query = """
SELECT TOP 100 *
FROM gaiadr3.vari_rrlyrae
WHERE pf IS NOT NULL
  AND num_clean_epochs_g > 40
ORDER BY num_clean_epochs_g DESC
"""


In [ ]:
rrlyrae = GaiaData(query, include_lightcurve=True)

rrlyrae.data[:10]


In [ ]:
TARGET_ID = 4659759557323962752
lightcurves = attach_flux_mean_magnitudes(rrlyrae.lightcurves)

axes = plot_raw_phase_folded_lightcurve(TARGET_ID, lightcurves, save=True)
plt.show()


In [ ]:
lightcurves = attach_periodogram_periods(lightcurves)

ax = plot_lomb_scargle_periodogram(TARGET_ID, lightcurves, save=True)
plt.show()


In [ ]:
source_ids, first_idx, counts = np.unique(
    lightcurves["source_id"],
    return_index=True,
    return_counts=True,
)
summary_rows = lightcurves[first_idx]

summary = table.Table(
    {
        "source_id": source_ids,
        "best_classification": summary_rows["best_classification"],
        "n_epochs": counts,
        "pf": summary_rows["pf"],
        "pf_error": summary_rows["pf_error"],
        "p1_o": summary_rows["p1_o"],
        "p1_o_error": summary_rows["p1_o_error"],
        "best_period": summary_rows["period_ls"],
        "mean_apparent_g": summary_rows["mean_g_transit_mag"],
        "mean_apparent_g_err": summary_rows["mean_g_transit_mag_err"],
        "int_average_g": summary_rows["int_average_g"],
        "int_average_g_error": summary_rows["int_average_g_error"],
    }
)



In [ ]:
axes = plot_vari_rrlyrae_period_comparison(summary, save=True)
plt.show()


In [ ]:
K_values = [1, 3, 5, 7, 9]

lightcurve = lightcurves[lightcurves["source_id"] == TARGET_ID]

axes = plot_fourier_harmonic_fits(lightcurve, K_values, save=True)
plt.show()


In [ ]:
cross_validation_res = cross_validate_harmonics(lightcurve)
best_K = cross_validation_res.best_K
period_ls = cross_validation_res.period

train_lightcurve = lightcurve[cross_validation_res.train_idx]
cv_lightcurve = lightcurve[cross_validation_res.cv_idx]

low_K = cross_validation_res.Ks[0]
high_K = cross_validation_res.Ks[-1]

low_fit = fourier_fit(train_lightcurve, period_ls, low_K)
best_fit = fourier_fit(train_lightcurve, period_ls, best_K)
high_fit = fourier_fit(train_lightcurve, period_ls, high_K)

ax = plot_fourier_cross_validation(cross_validation_res, save=True)
plt.show()


In [ ]:
axes = plot_fourier_cv_normalized_residual_histograms(train_lightcurve, cv_lightcurve, low_fit, best_fit, save=True)
plt.show()

axes = plot_fourier_cv_phase_comparison(train_lightcurve, cv_lightcurve, best_fit, high_fit, save=True)
plt.show()



In [ ]:
fit = fourier_fit(lightcurve, period_ls, best_K)
ax = plot_fourier_extrapolation(cross_validation_res, fit, save=True)
plt.show()


In [ ]:
source_ids = summary["source_id"]

best_k_by_star = np.full(len(source_ids), np.nan, dtype=float)
fourier_mean_g = np.full(len(source_ids), np.nan, dtype=float)
fourier_mean_g_err = np.full(len(source_ids), np.nan, dtype=float)
for i, source_id in enumerate(source_ids):
    star = lightcurves[lightcurves["source_id"] == source_id]
    period_i = float(summary["best_period"][summary["source_id"] == source_id][0])
    best_k_i = int(cross_validate_harmonics(star).best_K)
    fit = fourier_fit(star, period_i, best_k_i)
    best_k_by_star[i] = best_k_i
    fourier_mean_g[i] = fourier_mean_magnitude(fit)
    fourier_mean_g_err[i] = fourier_mean_magnitude_error(fit)

summary["best_K"] = best_k_by_star.astype(int)
summary["fourier_mean_apparent_g"] = fourier_mean_g
summary["fourier_mean_apparent_g_err"] = fourier_mean_g_err

axes = plot_mean_g_catalog_comparison(summary, save=True)
plt.show()

In [ ]:
# ============================================================
# 02-lightcurve-morphology.ipynb
# ============================================================

In [ ]:
import matplotlib.pyplot as plt

from ugdatalab import (
    GaiaData,
    plot_rrlyrae_shape_comparison,
)


In [ ]:
rrc_query = """
SELECT TOP 3 *
FROM gaiadr3.vari_rrlyrae
WHERE p1_o IS NOT NULL
  AND int_average_g  < 15
  AND num_clean_epochs_g > 80
ORDER BY num_clean_epochs_g DESC
"""


In [ ]:
rrc = GaiaData(rrc_query, include_lightcurve=True)


In [ ]:
rrab_query = """
SELECT TOP 3 *
FROM gaiadr3.vari_rrlyrae
WHERE pf IS NOT NULL
  AND p1_o IS NULL
  AND int_average_g  < 15
  AND num_clean_epochs_g > 80
ORDER BY num_clean_epochs_g DESC
"""


In [ ]:
rrab = GaiaData(rrab_query, include_lightcurve=True)


In [ ]:
axes = plot_rrlyrae_shape_comparison(rrab, rrc, save=True)
plt.show()

In [ ]:
# ============================================================
# 03-mcmc-validation.ipynb
# ============================================================

In [ ]:
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import plot_posterior, plot_trace

In [ ]:
x_obs = 1.0
sigma_obs = 0.1


def log_likelihood_1d(mu):
    return -0.5 * ((mu - x_obs) / sigma_obs) ** 2 - np.log(sigma_obs * np.sqrt(2 * np.pi))


def metropolis_hastings_1d(log_prob, theta0, proposal_std, n_steps, seed=42):
    rng = np.random.default_rng(seed)

    samples = np.empty((n_steps, 1))
    log_probs = np.empty(n_steps)

    theta = float(theta0)
    lp_curr = log_prob(theta)
    n_accepted = 0

    for i in range(n_steps):
        theta_prop = theta + rng.normal(0.0, proposal_std)
        lp_prop = log_prob(theta_prop)
        log_alpha = lp_prop - lp_curr

        if np.log(rng.uniform()) < log_alpha:
            theta = theta_prop
            lp_curr = lp_prop
            n_accepted += 1

        samples[i, 0] = theta
        log_probs[i] = lp_curr

    return SimpleNamespace(
        samples=samples,
        log_probs=log_probs,
        acceptance_rate=n_accepted / n_steps,
        n_burn=0,
        param_labels=[r"$\mu$"],
        proposal_std=proposal_std,
    )

In [ ]:
proposal_grid = [0.05, 0.10, 0.15, 0.18, 0.20, 0.25]
proposal_scan = []

for proposal_std in proposal_grid:
    test_chain = metropolis_hastings_1d(
        log_likelihood_1d,
        theta0=0.0,
        proposal_std=proposal_std,
        n_steps=4_000,
        seed=42,
    )
    proposal_scan.append(
        {
            "proposal_std": proposal_std,
            "acceptance_rate": round(test_chain.acceptance_rate, 3),
        }
    )

proposal_scan

In [ ]:
proposal_std = 0.20
n_steps = 10_000

mh = metropolis_hastings_1d(
    log_likelihood_1d,
    theta0=0.0,
    proposal_std=proposal_std,
    n_steps=n_steps,
    seed=42,
)

mu_samples = mh.samples[:, 0]

print(f"Acceptance rate: {mh.acceptance_rate:.3f}")
print(f"Sample mean of μ: {mu_samples.mean():.4f}   (analytic mean: {x_obs:.4f})")
print(f"Sample std of μ:  {mu_samples.std():.4f}   (analytic std:  {sigma_obs:.4f})")

In [ ]:
def analytic_pdf(mu):
    return np.exp(-0.5 * ((mu - x_obs) / sigma_obs) ** 2) / (sigma_obs * np.sqrt(2 * np.pi))


ax = plot_posterior(
    mh.samples,
    mh.param_labels,
    mh.n_burn,
    param_idx=0,
    pdf_fn=analytic_pdf,
    save_name="fig_mh_validation_posterior.pdf",
)
ax.set_title(r"Metropolis-Hastings samples vs. analytic $p(\mu \mid x{=}1,\,\sigma{=}0.1)$")
plt.show()


In [ ]:
axes = plot_trace(
    mh.samples,
    mh.log_probs,
    mh.param_labels,
    mh.n_burn,
)
axes[0].set_title(r"Trace: $\mu$ and $\ln P$ vs. step")
plt.show()


In [ ]:
# ============================================================
# 04-calibration-sample.ipynb
# ============================================================

In [ ]:
from astropy import table
from astropy.coordinates import SkyCoord
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import (
    GaiaData,
    GaiaQuality,
    Local,
    MixtureContaminationModel,
    pl_scatter_metrics,
    plot_inlier_prob_map,
    plot_inlier_prob_period_luminosity_comparison,
    plot_mollweide_diff,
    plot_period_abs_mag,
    plot_period_abs_mag_c12_comparison,
    rrlyrae_representative_period,
)

In [ ]:
query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""


In [ ]:
rrlyrae_quality = GaiaQuality(query)


In [ ]:
rrlyrae_low_dust = Local(rrlyrae_quality)
rrlyrae_low_dust_data = rrlyrae_low_dust.data

In [ ]:
assignment_style_query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
WHERE gs.parallax_over_error > 5
  AND ABS(gs.b) > 30
  AND gs.parallax > 0.25
"""

assignment_style_query

In [ ]:
len(rrlyrae_low_dust_data)


In [ ]:
abs_b = np.abs(rrlyrae_low_dust_data["b"])

float(np.min(abs_b)), int(np.count_nonzero(abs_b < 30.0))


In [ ]:
u = np.sqrt(
    rrlyrae_low_dust_data["astrometric_chi2_al"]
    / (rrlyrae_low_dust_data["astrometric_n_good_obs_al"] - 5)
)
G = np.asarray(rrlyrae_low_dust_data["phot_g_mean_mag"], dtype=float)
bp_rp = np.asarray(rrlyrae_low_dust_data["bp_rp"], dtype=float)
E = np.asarray(rrlyrae_low_dust_data["phot_bp_rp_excess_factor"], dtype=float)

cut1_mask = u < 1.2 * np.maximum(1.0, np.exp(-0.2 * (G - 19.5)))
cut2_mask = (E > 1.0 + 0.015 * bp_rp**2) & (E < 1.3 + 0.06 * bp_rp**2)
quality_mask = cut1_mask & cut2_mask

rrlyrae_calibration_data = rrlyrae_low_dust_data[quality_mask]

table.Table(
    rows=[
        {"sample": "Before C1/C2", "N": len(rrlyrae_low_dust_data)},
        {"sample": "Pass C1 only", "N": int(np.count_nonzero(cut1_mask))},
        {"sample": "Pass C2 only", "N": int(np.count_nonzero(cut2_mask))},
        {"sample": "Pass C1 and C2", "N": len(rrlyrae_calibration_data)},
    ]
)


In [ ]:
removed_c12 = rrlyrae_low_dust_data[~quality_mask]

ax = plot_mollweide_diff(rrlyrae_low_dust_data, rrlyrae_calibration_data, save=True)
ax.set_title(
    rf"Removed vs kept after C1/C2 ($N_{{\rm removed}}={len(removed_c12)}$, $N_{{\rm kept}}={len(rrlyrae_calibration_data)}$)"
)
plt.show()

removed_coords = SkyCoord(
    l=np.asarray(removed_c12["l"], dtype=float),
    b=np.asarray(removed_c12["b"], dtype=float),
    unit="deg",
    frame="galactic",
)
kept_coords = SkyCoord(
    l=np.asarray(rrlyrae_calibration_data["l"], dtype=float),
    b=np.asarray(rrlyrae_calibration_data["b"], dtype=float),
    unit="deg",
    frame="galactic",
)

lmc_center = SkyCoord(l=280.47, b=-32.89, unit="deg", frame="galactic")
smc_center = SkyCoord(l=302.81, b=-44.33, unit="deg", frame="galactic")

lmc_removed = removed_coords.separation(lmc_center).deg < 10.0
smc_removed = removed_coords.separation(smc_center).deg < 6.0
other_removed = ~(lmc_removed | smc_removed)

lmc_kept = kept_coords.separation(lmc_center).deg < 10.0
smc_kept = kept_coords.separation(smc_center).deg < 6.0
other_kept = ~(lmc_kept | smc_kept)

cluster_summary = table.Table(
    rows=[
        {
            "region": "LMC (<10 deg)",
            "N_removed": int(np.count_nonzero(lmc_removed)),
            "N_kept": int(np.count_nonzero(lmc_kept)),
            "Removed fraction": round(
                float(np.count_nonzero(lmc_removed))
                / max(
                    int(np.count_nonzero(lmc_removed)) + int(np.count_nonzero(lmc_kept)),
                    1,
                ),
                3,
            ),
        },
        {
            "region": "SMC (<6 deg)",
            "N_removed": int(np.count_nonzero(smc_removed)),
            "N_kept": int(np.count_nonzero(smc_kept)),
            "Removed fraction": round(
                float(np.count_nonzero(smc_removed))
                / max(
                    int(np.count_nonzero(smc_removed)) + int(np.count_nonzero(smc_kept)),
                    1,
                ),
                3,
            ),
        },
        {
            "region": "Elsewhere",
            "N_removed": int(np.count_nonzero(other_removed)),
            "N_kept": int(np.count_nonzero(other_kept)),
            "Removed fraction": round(
                float(np.count_nonzero(other_removed))
                / max(
                    int(np.count_nonzero(other_removed)) + int(np.count_nonzero(other_kept)),
                    1,
                ),
                3,
            ),
        },
    ]
)

cluster_summary

In [ ]:
ax = plot_period_abs_mag_c12_comparison(
    rrlyrae_low_dust_data,
    rrlyrae_calibration_data,
    save=True,
)
plt.show()


In [ ]:
scatter_metrics = table.Table(
    rows=[
        pl_scatter_metrics(rrlyrae_low_dust_data, "Before C1/C2"),
        pl_scatter_metrics(rrlyrae_calibration_data, "After C1/C2"),
    ]
)

scatter_metrics

In [ ]:
distance_kpc = 1.0 / rrlyrae_calibration_data["parallax"]

summary = table.Table(
    {
        "source_id": rrlyrae_calibration_data["source_id"],
        "best_classification": rrlyrae_calibration_data["best_classification"],
        "int_average_g": rrlyrae_calibration_data["int_average_g"],
        "b": rrlyrae_calibration_data["b"],
        "parallax": rrlyrae_calibration_data["parallax"],
        "parallax_error": rrlyrae_calibration_data["parallax_error"],
        "distance_kpc": distance_kpc,
    }
)

len(summary), summary[:10]


In [ ]:
rrlyrae_calibration = SimpleNamespace(
    query=query,
    include_lightcurve=False,
    data=rrlyrae_calibration_data,
    lightcurves=None,
)

rrlyrae_mixture_clean = MixtureContaminationModel(rrlyrae_calibration, prob_threshold=0.95)
rrlyrae_clean_data = rrlyrae_mixture_clean.data

table.Table(
    rows=[
        {"sample": "After C1/C2", "N": len(rrlyrae_calibration_data)},
        {"sample": rf"Mixture model ($p_{{\rm in}} \ge {rrlyrae_mixture_clean.prob_threshold:.2f}$)", "N": len(rrlyrae_clean_data)},
    ]
)


In [ ]:
cleaning_metrics = table.Table(
    rows=[
        pl_scatter_metrics(rrlyrae_calibration_data, "After C1/C2"),
        pl_scatter_metrics(rrlyrae_clean_data, "Mixture model"),
    ]
)

cleaning_metrics


In [ ]:
ax = plot_inlier_prob_period_luminosity_comparison(
    rrlyrae_mixture_clean,
    rrlyrae_calibration_data,
    rrlyrae_clean_data,
    save=True,
)
plt.show()


In [ ]:
ax = plot_inlier_prob_map(
    rrlyrae_mixture_clean,
    save=True,
)
ax.set_title(r"Calibration sample: inlier probability in the $P$--$M_G$ plane")
plt.show()

In [ ]:
distance_kpc = 1.0 / rrlyrae_clean_data["parallax"]

summary = table.Table(
    {
        "source_id": rrlyrae_clean_data["source_id"],
        "best_classification": rrlyrae_clean_data["best_classification"],
        "int_average_g": rrlyrae_clean_data["int_average_g"],
        "b": rrlyrae_clean_data["b"],
        "parallax": rrlyrae_clean_data["parallax"],
        "parallax_error": rrlyrae_clean_data["parallax_error"],
        "distance_kpc": distance_kpc,
    }
)

len(summary), summary[:10]


In [ ]:
ax = plot_period_abs_mag(
    np.asarray(rrlyrae_representative_period(rrlyrae_clean_data), dtype=float),
    np.asarray(rrlyrae_clean_data["M_G"], dtype=float),
    np.asarray(rrlyrae_clean_data["sigma_M"], dtype=float),
    np.asarray(rrlyrae_clean_data["best_classification"], dtype=str),
)
ax.set_title(rf"Final cleaned period-luminosity sample ($N={len(rrlyrae_clean_data)}$)")
plt.show()


In [ ]:
from pathlib import Path

from ugdatalab import save_table_npz

rrlyrae_export_data = rrlyrae_clean_data.copy()
output_path = Path("rrlyrae_calibration_sample.npz")
save_table_npz(output_path, rrlyrae_export_data)

{
    "path": str(output_path.resolve()),
    "N": len(rrlyrae_export_data),
    "columns": len(rrlyrae_export_data.colnames),
}


In [ ]:
# ============================================================
# 05-optical-pl-fits.ipynb
# ============================================================

In [ ]:
from pathlib import Path
from types import SimpleNamespace

from astropy import table
import corner
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt

from ugdatalab import (
    MetropolisHastings,
    NoUTurnHamiltonian,
    plot_pl_sampler_comparison_corner,
    build_optical_pl_comparison_data,
    build_pl_context,
    fit_pl_nuts_native,
    fit_pl_nuts_potential,
    load_table_npz,
    pl_posterior_summary_row,
    plot_corner,
    plot_pl_posterior_predictive,
    plot_pl_posterior_predictive_comparison,
    mcmc_sampler_color,
    plot_trace,
    save_optical_pl_comparison_data,
)

In [ ]:
data_path = Path("rrlyrae_calibration_sample.npz")
rrlyrae_clean_data = load_table_npz(data_path)

len(rrlyrae_clean_data), rrlyrae_clean_data.colnames[:8]


In [ ]:
classes = np.asarray(rrlyrae_clean_data["best_classification"]).astype(str)
class_labels, class_counts = np.unique(classes, return_counts=True)
class_count_table = table.Table({"class": class_labels, "N": class_counts})

rrab_data = rrlyrae_clean_data[classes == "RRab"]
rrc_data = rrlyrae_clean_data[classes == "RRc"]
rrd_data = rrlyrae_clean_data[classes == "RRd"]

class_count_table


In [ ]:
DISPLAY_LABELS = [r"$a$", r"$b$", r"$\sigma_{\rm scatter}$"]
SLOPE_PRIOR_SIGMA = 5.0
INTERCEPT_PRIOR_SIGMA = 1.0
SIGMA_SCATTER_SCALE = 0.5
MH_PROPOSAL_GRID = [
    np.array([0.004, 0.008, 0.008]),
    np.array([0.005, 0.010, 0.010]),
    np.array([0.006, 0.012, 0.012]),
    np.array([0.008, 0.016, 0.016]),
    np.array([0.010, 0.020, 0.020]),
]


def log_prior_mh(theta, ctx):
    a, b, log10_sigma = theta
    if not np.all(np.isfinite(theta)):
        return -np.inf

    sigma_scatter = 10.0 ** log10_sigma
    if sigma_scatter <= 0.0:
        return -np.inf

    logp_a = -0.5 * (a / SLOPE_PRIOR_SIGMA) ** 2 - np.log(SLOPE_PRIOR_SIGMA * np.sqrt(2.0 * np.pi))
    logp_b = (
        -0.5 * ((b - ctx.intercept_prior_mean) / INTERCEPT_PRIOR_SIGMA) ** 2
        - np.log(INTERCEPT_PRIOR_SIGMA * np.sqrt(2.0 * np.pi))
    )
    logp_sigma = (
        0.5 * np.log(2.0 / np.pi)
        - np.log(SIGMA_SCATTER_SCALE)
        - 0.5 * (sigma_scatter / SIGMA_SCATTER_SCALE) ** 2
        + np.log(np.log(10.0) * sigma_scatter)
    )
    return logp_a + logp_b + logp_sigma


def log_likelihood_pl(theta, ctx):
    a, b, log10_sigma = theta
    sigma_scatter = 10.0 ** log10_sigma
    sigma2_tot = ctx.sigma**2 + sigma_scatter**2 + (a * ctx.sigma_logp) ** 2
    mu = a * ctx.x_centered + b
    return -0.5 * np.sum(np.log(2.0 * np.pi * sigma2_tot) + (ctx.y - mu) ** 2 / sigma2_tot)


def log_posterior_pl(theta, ctx):
    log_prior = log_prior_mh(theta, ctx)
    if not np.isfinite(log_prior):
        return -np.inf
    return log_prior + log_likelihood_pl(theta, ctx)


def build_display_sampler(samples, log_probs, n_burn):
    display_samples = np.asarray(samples, dtype=float).copy()
    display_samples[:, 2] = 10.0 ** display_samples[:, 2]
    return SimpleNamespace(
        samples=display_samples,
        log_probs=np.asarray(log_probs, dtype=float).copy(),
        n_burn=n_burn,
        param_labels=DISPLAY_LABELS,
    )


def proposal_scan(ctx):
    rows = []
    for proposal_std in MH_PROPOSAL_GRID:
        pilot = MetropolisHastings(
            log_prob=lambda theta, c=ctx: log_posterior_pl(theta, c),
            theta0=ctx.theta0,
            proposal_std=proposal_std,
            seed=42,
            labels=[r"$a$", r"$b$", r"$\log_{10}\sigma_{\rm scatter}$"],
        )
        pilot.run(n_steps=4_000, n_burn=500)
        rows.append(
            {
                "class": ctx.class_label,
                "proposal_std": proposal_std.tolist(),
                "acceptance": round(float(pilot.acceptance_rate), 3),
            }
        )
    return table.Table(rows=rows)


def run_mh_fit(ctx, proposal_std, n_steps=25_000, n_burn=5_000):
    sampler_raw = MetropolisHastings(
        log_prob=lambda theta, c=ctx: log_posterior_pl(theta, c),
        theta0=ctx.theta0,
        proposal_std=proposal_std,
        seed=42,
        labels=[r"$a$", r"$b$", r"$\log_{10}\sigma_{\rm scatter}$"],
    )
    sampler_raw.run(n_steps=n_steps, n_burn=n_burn)
    sampler_display = build_display_sampler(sampler_raw.samples, sampler_raw.log_probs, sampler_raw.n_burn)
    post_burn_samples = sampler_display.samples[sampler_display.n_burn:]
    summary = pl_posterior_summary_row("Metropolis-Hastings", ctx.class_label, post_burn_samples, sampler_raw.acceptance_rate)
    return SimpleNamespace(raw=sampler_raw, display=sampler_display, samples=post_burn_samples, summary=summary)


rrab_data = rrlyrae_clean_data[np.asarray(rrlyrae_clean_data["best_classification"]).astype(str) == "RRab"]
rrc_data = rrlyrae_clean_data[np.asarray(rrlyrae_clean_data["best_classification"]).astype(str) == "RRc"]
rrd_data = rrlyrae_clean_data[np.asarray(rrlyrae_clean_data["best_classification"]).astype(str) == "RRd"]

rrab_ctx = build_pl_context(rrab_data, "RRab")
rrc_ctx = build_pl_context(rrc_data, "RRc")
print({"RRab": rrab_ctx.n, "RRc": rrc_ctx.n, "RRd excluded": len(rrd_data)})

In [ ]:
rrab_proposal_scan = proposal_scan(rrab_ctx)
rrab_proposal_std = np.array(
    min(rrab_proposal_scan, key=lambda row: abs(float(row["acceptance"]) - 0.5))["proposal_std"],
    dtype=float,
)
rrab_proposal_scan


In [ ]:
rrab_mh = run_mh_fit(rrab_ctx, rrab_proposal_std)
rrab_mh.summary


In [ ]:
axes = plot_trace(
    rrab_mh.display.samples,
    rrab_mh.display.log_probs,
    rrab_mh.display.param_labels,
    rrab_mh.display.n_burn,
    color=mcmc_sampler_color("RRab", "mh"),
)
axes[0].set_title('Trace: RRab G-band PL fit (Metropolis-Hastings)')
plt.show()

In [ ]:
fig = plot_corner(
    rrab_mh.display,
    color=mcmc_sampler_color("RRab", "mh"),
    save="fig_mh_corner.pdf",
)
fig.suptitle(r'Posterior: RRab $a$, $b$, and $\sigma_{\rm scatter}$ (Metropolis-Hastings)')
plt.show()

In [ ]:
rrab_nuts_potential = fit_pl_nuts_potential(rrab_ctx)
rrab_nuts_potential.summary = pl_posterior_summary_row("NUTS + Potential", rrab_ctx.class_label, rrab_nuts_potential.samples, rrab_nuts_potential.acceptance_rate)
rrab_nuts_potential.summary

In [ ]:
axes = plot_trace(
    rrab_nuts_potential.display.samples,
    rrab_nuts_potential.display.log_probs,
    rrab_nuts_potential.display.param_labels,
    rrab_nuts_potential.display.n_burn,
    color=mcmc_sampler_color("RRab", "nuts + potential"),
)
axes[0].set_title('Trace: RRab G-band PL fit (NUTS + Potential)')
plt.show()

In [ ]:
fig = plot_corner(
    rrab_nuts_potential.display,
    color=mcmc_sampler_color("RRab", "nuts + potential"),
    save="fig_nuts_corner.pdf",
)
fig.suptitle(r'Posterior: RRab $a$, $b$, and $\sigma_{\rm scatter}$ (NUTS + Potential)')
plt.show()

In [ ]:
rrab_nuts_native = fit_pl_nuts_native(rrab_ctx)
rrab_nuts_native.summary = pl_posterior_summary_row("Native PyMC NUTS", rrab_ctx.class_label, rrab_nuts_native.samples, rrab_nuts_native.acceptance_rate)
rrab_nuts_native.summary

In [ ]:
axes = plot_trace(
    rrab_nuts_native.display.samples,
    rrab_nuts_native.display.log_probs,
    rrab_nuts_native.display.param_labels,
    rrab_nuts_native.display.n_burn,
    color=mcmc_sampler_color("RRab", "native"),
)
axes[0].set_title('Trace: RRab G-band PL fit (native PyMC NUTS)')
plt.show()

In [ ]:
fig = plot_corner(
    rrab_nuts_native.display,
    color=mcmc_sampler_color("RRab", "native"),
    save="fig_native_corner.pdf",
)
fig.suptitle(r'Posterior: RRab $a$, $b$, and $\sigma_{\rm scatter}$ (native PyMC NUTS)')
plt.show()

In [ ]:
rrab_summary = table.Table(rows=[rrab_mh.summary, rrab_nuts_potential.summary, rrab_nuts_native.summary])
rrab_summary


In [ ]:
fig = plot_pl_sampler_comparison_corner(
    {
        "Metropolis-Hastings": rrab_mh.display.samples[rrab_mh.display.n_burn:],
        "NUTS + Potential": rrab_nuts_potential.display.samples[rrab_nuts_potential.display.n_burn:],
        "Native PyMC NUTS": rrab_nuts_native.display.samples[rrab_nuts_native.display.n_burn:],
    },
    class_label="RRab",
    save_name="fig_methods_corner_rrab.pdf",
)
fig.suptitle(r"RRab: posterior comparison across all three MCMC methods", y=1.01)
plt.show()

In [ ]:
ax = plot_pl_posterior_predictive(
    rrab_ctx,
    rrab_nuts_native.samples[rrab_nuts_native.display.n_burn:],
    save="fig_pl_posterior.pdf",
)
ax.set_title('RRab posterior predictive check (native PyMC NUTS)')
plt.show()

In [ ]:
rrc_proposal_scan = proposal_scan(rrc_ctx)
rrc_proposal_std = np.array(
    min(rrc_proposal_scan, key=lambda row: abs(float(row["acceptance"]) - 0.5))["proposal_std"],
    dtype=float,
)
rrc_proposal_scan


In [ ]:
rrc_mh = run_mh_fit(rrc_ctx, rrc_proposal_std)
rrc_mh.summary


In [ ]:
axes = plot_trace(
    rrc_mh.display.samples,
    rrc_mh.display.log_probs,
    rrc_mh.display.param_labels,
    rrc_mh.display.n_burn,
    color=mcmc_sampler_color("RRc", "mh"),
)
axes[0].set_title('Trace: RRc G-band PL fit (Metropolis-Hastings)')
plt.show()

In [ ]:
fig = plot_corner(
    rrc_mh.display,
    color=mcmc_sampler_color("RRc", "mh"),
)
fig.suptitle(r'Posterior: RRc $a$, $b$, and $\sigma_{\rm scatter}$ (Metropolis-Hastings)')
plt.show()

In [ ]:
rrc_nuts_potential = fit_pl_nuts_potential(rrc_ctx)
rrc_nuts_potential.summary = pl_posterior_summary_row("NUTS + Potential", rrc_ctx.class_label, rrc_nuts_potential.samples, rrc_nuts_potential.acceptance_rate)
rrc_nuts_potential.summary

In [ ]:
axes = plot_trace(
    rrc_nuts_potential.display.samples,
    rrc_nuts_potential.display.log_probs,
    rrc_nuts_potential.display.param_labels,
    rrc_nuts_potential.display.n_burn,
    color=mcmc_sampler_color("RRc", "nuts + potential"),
)
axes[0].set_title('Trace: RRc G-band PL fit (NUTS + Potential)')
plt.show()

In [ ]:
fig = plot_corner(
    rrc_nuts_potential.display,
    color=mcmc_sampler_color("RRc", "nuts + potential"),
)
fig.suptitle(r'Posterior: RRc $a$, $b$, and $\sigma_{\rm scatter}$ (NUTS + Potential)')
plt.show()

In [ ]:
rrc_nuts_native = fit_pl_nuts_native(rrc_ctx)
rrc_nuts_native.summary = pl_posterior_summary_row("Native PyMC NUTS", rrc_ctx.class_label, rrc_nuts_native.samples, rrc_nuts_native.acceptance_rate)
rrc_nuts_native.summary

In [ ]:
axes = plot_trace(
    rrc_nuts_native.display.samples,
    rrc_nuts_native.display.log_probs,
    rrc_nuts_native.display.param_labels,
    rrc_nuts_native.display.n_burn,
    color=mcmc_sampler_color("RRc", "native"),
)
axes[0].set_title('Trace: RRc G-band PL fit (native PyMC NUTS)')
plt.show()

In [ ]:
fig = plot_corner(
    rrc_nuts_native.display,
    color=mcmc_sampler_color("RRc", "native"),
)
fig.suptitle(r'Posterior: RRc $a$, $b$, and $\sigma_{\rm scatter}$ (native PyMC NUTS)')
plt.show()

In [ ]:
rrc_summary = table.Table(rows=[rrc_mh.summary, rrc_nuts_potential.summary, rrc_nuts_native.summary])
rrc_summary


In [ ]:
fig = plot_pl_sampler_comparison_corner(
    {
        "Metropolis-Hastings": rrc_mh.display.samples[rrc_mh.display.n_burn:],
        "NUTS + Potential": rrc_nuts_potential.display.samples[rrc_nuts_potential.display.n_burn:],
        "Native PyMC NUTS": rrc_nuts_native.display.samples[rrc_nuts_native.display.n_burn:],
    },
    class_label="RRc",
    save_name="fig_methods_corner_rrc.pdf",
)
fig.suptitle(r"RRc: posterior comparison across all three MCMC methods", y=1.01)
plt.show()

In [ ]:
ax = plot_pl_posterior_predictive(
    rrc_ctx,
    rrc_nuts_native.samples[rrc_nuts_native.display.n_burn:],
    save="fig_pl_posterior_rrc.pdf",
)
ax.set_title('RRc posterior predictive check (native PyMC NUTS)')
plt.show()

In [ ]:
comparison_summary = table.Table(
    rows=[
        rrab_nuts_native.summary,
        rrc_nuts_native.summary,
    ]
)
comparison_summary


In [ ]:
ax = plot_pl_posterior_predictive_comparison(
    rrab_ctx,
    rrab_nuts_native.samples,
    rrc_ctx,
    rrc_nuts_native.samples,
)
ax.set_title('RRab and RRc native-PyMC posterior predictive comparison')
plt.show()


In [ ]:
output_path = Path("rrlyrae_optical_pl_comparison_data.npz")
comparison_export = {
    "RRab": build_optical_pl_comparison_data(rrab_ctx, rrab_nuts_native.samples),
    "RRc": build_optical_pl_comparison_data(rrc_ctx, rrc_nuts_native.samples),
}
save_optical_pl_comparison_data(output_path, comparison_export)

{
    "path": str(output_path.resolve()),
    "classes": list(comparison_export),
}


In [ ]:
# ============================================================
# 06-wise-pl-literature.ipynb
# ============================================================

In [ ]:
from pathlib import Path
from types import SimpleNamespace

from astropy import table
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import (
    GaiaData,
    WISEQualityFilter,
    attach_w2_absolute_magnitude,
    build_band_summary,
    build_infrared_pl_comparison_data,
    build_w2_context,
    fit_w2_nuts,
    load_optical_pl_comparison_data,
    load_or_create_table_npz,
    load_table_npz,
    mcmc_sampler_color,
    plot_corner,
    plot_mollweide_diff,
    plot_optical_vs_w2_comparison,
    plot_period_abs_mag,
    plot_period_abs_mag_comparison,
    plot_trace,
    plot_w2_posterior_predictive,
    rrlyrae_representative_period,
    save_infrared_pl_comparison_data,
    w2_posterior_summary_row,
)

In [ ]:
DISPLAY_LABELS = [r"$a$", r"$b$", r"$\sigma_{\rm scatter}$"]
SLOPE_PRIOR_SIGMA = 5.0
INTERCEPT_PRIOR_SIGMA = 1.0
SIGMA_SCATTER_SCALE = 0.5
NUTS_DRAWS = 2_000
NUTS_TUNE = 1_000

clean_path = Path("rrlyrae_calibration_sample.npz")
optical_path = Path("rrlyrae_optical_pl_comparison_data.npz")
gaia_wise_cache_path = Path("rrlyrae_gaia_wise_query_data.npz")

infrared_path = Path("rrlyrae_infrared_pl_comparison_data.npz")


In [ ]:
query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
LEFT OUTER JOIN gaiadr3.allwise_best_neighbour AS bn
    ON vr.source_id = bn.source_id
LEFT OUTER JOIN gaiadr3.allwise_neighbourhood AS nh
    ON bn.source_id = nh.source_id
   AND bn.allwise_oid = nh.allwise_oid
LEFT OUTER JOIN gaiadr1.allwise_original_valid AS aw
    ON bn.allwise_oid = aw.allwise_oid
"""


In [ ]:
def _fetch_gaia_wise_table():
    return GaiaData(query).data


gaia_wise_data, cache_status = load_or_create_table_npz(
    gaia_wise_cache_path,
    _fetch_gaia_wise_table,
)

{
    "cache": str(gaia_wise_cache_path.resolve()),
    "status": cache_status,
    "rows": len(gaia_wise_data),
}


In [ ]:
len(gaia_wise_data)


In [ ]:
clean_sample = load_table_npz(clean_path)
clean_source_ids = np.asarray(clean_sample["source_id"], dtype=np.int64)

clean_mask = np.isin(np.asarray(gaia_wise_data["source_id"], dtype=np.int64), clean_source_ids)
clean_gaia_wise_data = gaia_wise_data[clean_mask]

print({
    "clean source_id count": len(clean_source_ids),
    "rows recovered from Gaia+WISE query": len(clean_gaia_wise_data),
    "missing source_ids": int(len(clean_source_ids) - len(clean_gaia_wise_data)),
})


In [ ]:
optical_comparison = load_optical_pl_comparison_data(optical_path)
optical_comparison


In [ ]:
def _col_as_float(column):
    values = np.ma.asarray(column, dtype=float)
    return np.asarray(np.ma.filled(values, np.nan), dtype=float)


def _col_as_str(column):
    values = np.ma.asarray(column, dtype=object)
    filled = np.ma.filled(values, "")
    return np.asarray([str(v).strip() if v is not None else "" for v in filled], dtype=str)


def _char_at(column, idx):
    strings = _col_as_str(column)
    return np.asarray([value[idx] if len(value) > idx else "" for value in strings], dtype=str)


def _first_present_float(data, names):
    for name in names:
        if name in data.colnames:
            return _col_as_float(data[name])
    raise KeyError(f"Missing all candidate columns: {names}")

In [ ]:
clean_gaia_wise_data = attach_w2_absolute_magnitude(clean_gaia_wise_data)

allwise_oid = _col_as_float(clean_gaia_wise_data["allwise_oid"])
matched_best_neighbour_data = clean_gaia_wise_data[np.isfinite(allwise_oid)]

rr_classes = _col_as_str(matched_best_neighbour_data["best_classification"])
matched_best_neighbour_rr_data = matched_best_neighbour_data[np.isin(rr_classes, ["RRab", "RRc", "RRd"])]

wise_clean_data = WISEQualityFilter(matched_best_neighbour_rr_data).data

stage_counts = table.Table(
    rows=[
        {"stage": "Clean Gaia sample", "N": len(clean_gaia_wise_data)},
        {"stage": "With any WISE best-neighbour match", "N": len(matched_best_neighbour_data)},
        {"stage": "Matched RRab/RRc/RRd subset", "N": len(matched_best_neighbour_rr_data)},
        {"stage": "Final conservative WISE-quality subset", "N": len(wise_clean_data)},
    ]
)

matched_class_counts = {
    label: int(np.count_nonzero(_col_as_str(matched_best_neighbour_rr_data["best_classification"]) == label))
    for label in ("RRab", "RRc", "RRd")
}
wise_class_counts = {
    label: int(np.count_nonzero(_col_as_str(wise_clean_data["best_classification"]) == label))
    for label in ("RRab", "RRc", "RRd")
}
class_counts = table.Table(
    rows=[
        {
            "class": label,
            "best_neighbour_matches": matched_class_counts[label],
            "final_wise_quality": wise_class_counts[label],
        }
        for label in ("RRab", "RRc", "RRd")
    ]
)

stage_counts, class_counts

In [ ]:
stage_counts


In [ ]:
class_counts


In [ ]:
ax = plot_mollweide_diff(clean_gaia_wise_data, matched_best_neighbour_data)
ax.set_title("Clean Gaia sample vs. stars with any WISE best-neighbour match")
plt.show()


In [ ]:
ax = plot_mollweide_diff(clean_gaia_wise_data, wise_clean_data)
ax.set_title("Clean Gaia sample vs. final conservative WISE-quality subset")
plt.show()


In [ ]:
matched_period = rrlyrae_representative_period(matched_best_neighbour_rr_data)
wise_period = rrlyrae_representative_period(wise_clean_data)
matched_classes = _col_as_str(matched_best_neighbour_rr_data["best_classification"])
wise_classes = _col_as_str(wise_clean_data["best_classification"])

axes = plot_period_abs_mag_comparison(
    matched_period,
    _col_as_float(matched_best_neighbour_rr_data["M_W2"]),
    _col_as_float(matched_best_neighbour_rr_data["sigma_M_W2"]),
    matched_classes,
    wise_period,
    _col_as_float(wise_clean_data["M_W2"]),
    _col_as_float(wise_clean_data["sigma_M_W2"]),
    wise_classes,
)
axes[0].set_title("All clean Gaia RR Lyrae with a WISE best-neighbour match")
axes[1].set_title("Final conservative WISE-quality subset")
for ax in axes:
    ax.set_ylabel(r"$M_{W2}$ [mag]")
plt.show()


In [ ]:
ax = plot_period_abs_mag(
    wise_period,
    _col_as_float(wise_clean_data["M_W2"]),
    _col_as_float(wise_clean_data["sigma_M_W2"]),
    wise_classes,
)
ax.set_ylabel(r"$M_{W2}$ [mag]")
ax.set_title(rf"Final WISE quality subset ($N={len(wise_clean_data)}$)")
plt.show()

In [ ]:
wise_rrab_data = wise_clean_data[_col_as_str(wise_clean_data["best_classification"]) == "RRab"]
wise_rrc_data = wise_clean_data[_col_as_str(wise_clean_data["best_classification"]) == "RRc"]
wise_rrd_data = wise_clean_data[_col_as_str(wise_clean_data["best_classification"]) == "RRd"]

rrab_ctx = build_w2_context(wise_rrab_data, "RRab")
rrc_ctx = build_w2_context(wise_rrc_data, "RRc")

table.Table(
    rows=[
        {"class": "RRab", "N_fit": rrab_ctx.n},
        {"class": "RRc", "N_fit": rrc_ctx.n},
        {"class": "RRd excluded", "N": len(wise_rrd_data)},
    ]
)


In [ ]:
rrab_sampler = fit_w2_nuts(rrab_ctx, seed=128)
rrab_sampler.acceptance_rate

In [ ]:
rrab_summary = table.Table(rows=[w2_posterior_summary_row("RRab", "WISE $W2$", rrab_sampler, rrab_ctx.n)])
rrab_summary

In [ ]:
axes = plot_trace(
    rrab_sampler.samples,
    rrab_sampler.log_probs,
    rrab_sampler.param_labels,
    rrab_sampler.n_burn,
    color=mcmc_sampler_color("RRab", "native"),
)
axes[0].figure.suptitle("RRab: NUTS trace diagnostics in WISE $W2$", y=1.01)
plt.show()

In [ ]:
fig = plot_corner(
    rrab_sampler,
    labels=DISPLAY_LABELS,
    color=mcmc_sampler_color("RRab", "native"),
)
fig.suptitle("RRab: posterior constraints in WISE $W2$", y=1.02)
plt.show()

In [ ]:
ax = plot_w2_posterior_predictive(
    rrab_ctx,
    rrab_sampler.samples[rrab_sampler.n_burn:],
    save_name="fig_wise_pl_rrab.pdf",
)
plt.show()

In [ ]:
rrc_sampler = fit_w2_nuts(rrc_ctx, seed=256)
rrc_sampler.acceptance_rate

In [ ]:
rrc_summary = table.Table(rows=[w2_posterior_summary_row("RRc", "WISE $W2$", rrc_sampler, rrc_ctx.n)])
rrc_summary

In [ ]:
axes = plot_trace(
    rrc_sampler.samples,
    rrc_sampler.log_probs,
    rrc_sampler.param_labels,
    rrc_sampler.n_burn,
    color=mcmc_sampler_color("RRc", "native"),
)
axes[0].figure.suptitle("RRc: NUTS trace diagnostics in WISE $W2$", y=1.01)
plt.show()

In [ ]:
fig = plot_corner(
    rrc_sampler,
    labels=DISPLAY_LABELS,
    color=mcmc_sampler_color("RRc", "native"),
)
fig.suptitle("RRc: posterior constraints in WISE $W2$", y=1.02)
plt.show()

In [ ]:
ax = plot_w2_posterior_predictive(
    rrc_ctx,
    rrc_sampler.samples[rrc_sampler.n_burn:],
    save_name="fig_wise_pl_rrc.pdf",
)
plt.show()

In [ ]:
rrab_w2 = build_infrared_pl_comparison_data(rrab_ctx, rrab_sampler.samples)
rrc_w2 = build_infrared_pl_comparison_data(rrc_ctx, rrc_sampler.samples)
wise_comparison = {"RRab": rrab_w2, "RRc": rrc_w2}

comparison_rows = []
for class_label in ("RRab", "RRc"):
    optical = optical_comparison[class_label]
    wise = wise_comparison[class_label]
    comparison_rows.extend(
        [
            {
                "class": class_label,
                "band": "Gaia $G$",
                "N_fit": len(optical.x_obs),
                "a": round(float(optical.slope_q50), 4),
                "a_minus": round(float(optical.slope_q50 - optical.slope_q16), 4),
                "a_plus": round(float(optical.slope_q84 - optical.slope_q50), 4),
                "sigma_scatter": round(float(optical.sigma_scatter_q50), 4),
                "sigma_minus": round(float(optical.sigma_scatter_q50 - optical.sigma_scatter_q16), 4),
                "sigma_plus": round(float(optical.sigma_scatter_q84 - optical.sigma_scatter_q50), 4),
            },
            {
                "class": class_label,
                "band": "WISE $W2$",
                "N_fit": len(wise.x_obs),
                "a": round(float(wise.slope_q50), 4),
                "a_minus": round(float(wise.slope_q50 - wise.slope_q16), 4),
                "a_plus": round(float(wise.slope_q84 - wise.slope_q50), 4),
                "sigma_scatter": round(float(wise.sigma_scatter_q50), 4),
                "sigma_minus": round(float(wise.sigma_scatter_q50 - wise.sigma_scatter_q16), 4),
                "sigma_plus": round(float(wise.sigma_scatter_q84 - wise.sigma_scatter_q50), 4),
            },
        ]
    )

comparison_table = table.Table(rows=comparison_rows)
comparison_table


In [ ]:
axes = plot_optical_vs_w2_comparison(optical_comparison, wise_comparison, save=True)
plt.show()


In [ ]:
rrab_optical = optical_comparison["RRab"]
rrc_optical = optical_comparison["RRc"]

rrab_w2_steeper = abs(rrab_w2.slope_q50) > abs(rrab_optical.slope_q50)
rrc_w2_steeper = abs(rrc_w2.slope_q50) > abs(rrc_optical.slope_q50)
rrab_w2_tighter = rrab_w2.sigma_scatter_q50 < rrab_optical.sigma_scatter_q50
rrc_w2_tighter = rrc_w2.sigma_scatter_q50 < rrc_optical.sigma_scatter_q50

analysis = f"""
## Analysis and Discussion

The infrared comparison behaves in the direction expected from the RR Lyrae literature. In the optical, the Gaia $G$ relation is broadened by the broad-band response to temperature changes across the pulsation cycle, residual extinction, and population effects; by contrast, near-infrared relations are expected to be less dust-sensitive and more nearly luminosity-dominated, which makes them both tighter and often steeper ([Klein et al. 2014](https://academic.oup.com/mnrasl/article/440/1/L96/1396776); [Beaton et al. 2018](https://arxiv.org/abs/1805.01552)).

For **RRab**, this notebook finds a Gaia $G$ slope of **{rrab_optical.slope_q50:.3f}** and a WISE $W2$ slope of **{rrab_w2.slope_q50:.3f}**. In absolute value, the WISE relation is {'larger' if rrab_w2_steeper else 'smaller'} than the optical one, so the RRab $W2$ relation is {'steeper' if rrab_w2_steeper else 'not steeper'} in this fit. The inferred intrinsic scatter also goes from **{rrab_optical.sigma_scatter_q50:.3f} mag** in Gaia $G$ to **{rrab_w2.sigma_scatter_q50:.3f} mag** in WISE $W2$, so the RRab infrared relation is {'tighter' if rrab_w2_tighter else 'not tighter'} than the optical one.

For **RRc**, the Gaia $G$ slope is **{rrc_optical.slope_q50:.3f}** and the WISE $W2$ slope is **{rrc_w2.slope_q50:.3f}**. In absolute value, the WISE relation is {'larger' if rrc_w2_steeper else 'smaller'} than the optical one, so the RRc $W2$ relation is {'steeper' if rrc_w2_steeper else 'not steeper'} in this fit. The inferred intrinsic scatter changes from **{rrc_optical.sigma_scatter_q50:.3f} mag** in Gaia $G$ to **{rrc_w2.sigma_scatter_q50:.3f} mag** in WISE $W2$, so the RRc infrared relation is {'tighter' if rrc_w2_tighter else 'not tighter'} than the optical one.

The predictive-envelope plots are the most important visual check. They show the full model spread implied by the measurement errors, the propagated period uncertainties, and the fitted intrinsic scatter. If the model is behaving sensibly, the observed points should sit inside the predictive envelopes at roughly the rate one expects for a 68% band, without the systematic subclass-dependent mismatch that would signal a poor bandpass model or an over-optimistic scatter term.

The main caveat is the same as in `02-02.ipynb`: this notebook still works in absolute-magnitude space after inverse-parallax conversion. That is pedagogically convenient and keeps the optical and infrared comparison easy to follow, but a research-grade final calibration would more naturally be phrased in parallax space or in a hierarchical Bayesian model ([Gaia Collaboration et al. 2017](https://www.aanda.org/articles/aa/full_html/2017/09/aa29925-16/aa29925-16.html); [Luri et al. 2018](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html); [Muraveva et al. 2018](https://academic.oup.com/mnras/article/481/1/1195/5075596)). Within that limitation, the physically interesting conclusion is the slope comparison: the steeper band is the one with the larger fitted **$|a|$**, and in this notebook that answer is determined directly by the RRab and RRc posterior medians reported above.

### Sources
- Klein et al. (2014), *A mid-infrared period-luminosity relation for RR Lyrae stars*. https://academic.oup.com/mnrasl/article/440/1/L96/1396776
- Beaton et al. (2018), *The Carnegie-Chicago Hubble Program*. https://arxiv.org/abs/1805.01552
- Gaia Collaboration et al. (2017), *Gaia parallaxes and period-luminosity relations*. https://www.aanda.org/articles/aa/full_html/2017/09/aa29925-16/aa29925-16.html
- Luri et al. (2018), *Using Gaia parallaxes*. https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html
- Muraveva et al. (2018), *Bayesian RR Lyrae calibration with Gaia parallaxes*. https://academic.oup.com/mnras/article/481/1/1195/5075596
- Narloch et al. (2024), *On the use of RR Lyrae stars for distance determination*. https://www.aanda.org/articles/aa/full_html/2024/09/aa50364-24/aa50364-24.html
"""

display(Markdown(analysis))


In [ ]:
infrared_export = {
    "RRab": rrab_w2,
    "RRc": rrc_w2,
}
save_infrared_pl_comparison_data(infrared_path, infrared_export)

{
    "path": str(infrared_path.resolve()),
    "classes": list(infrared_export),
}


In [ ]:
import io
import os
from contextlib import redirect_stdout
from pathlib import Path
from IPython.display import Markdown, display

cache_root = Path('/tmp/ay128_lab01_cache')
cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'mpl'))
os.environ.setdefault('XDG_CACHE_HOME', str(cache_root / 'xdg'))
os.environ.setdefault('PYTENSOR_FLAGS', f"base_compiledir={cache_root / 'pytensor'}")

from astropy import table
import numpy as np

with redirect_stdout(io.StringIO()):
    from ugdatalab import (
        load_infrared_pl_comparison_data,
        load_optical_pl_comparison_data,
    )

optical_path = Path('rrlyrae_optical_pl_comparison_data.npz')
infrared_path = Path('rrlyrae_infrared_pl_comparison_data.npz')

optical_data = load_optical_pl_comparison_data(optical_path)
infrared_data = load_infrared_pl_comparison_data(infrared_path)

literature = {
    'klein_optical': {
        'statement': 'In optical wavebands the RR Lyrae period-luminosity relation is negligible, but the slope steepens strongly in the infrared.',
        'source': 'Klein et al. 2014, Introduction',
        'url': 'https://academic.oup.com/mnrasl/article/440/1/L96/1396776',
    },
    'beaton_review': {
        'statement': 'RR Lyrae calibration depends on wavelength, reddening, metallicity, and how the absolute scale is established.',
        'source': 'Beaton et al. 2018',
        'url': 'https://doi.org/10.1007/s11214-018-0542-1',
    },
    'klein_w2': {
        'RRab': {'slope': -2.39, 'err': 0.20, 'pivot': 0.55},
        'RRc': {'slope': -1.70, 'err': 0.62, 'pivot': 0.32},
        'url': 'https://academic.oup.com/mnrasl/article/440/1/L96/1396776',
    },
    'dambis_w2': {
        'slope': -2.269,
        'err': 0.127,
        'metallicity_coeff': 0.108,
        'metallicity_err': 0.021,
        'url': 'https://doi.org/10.1093/mnras/stu226',
    },
}

{
    'optical_artifact': str(optical_path.resolve()),
    'infrared_artifact': str(infrared_path.resolve()),
    'RRab_G_points': int(len(optical_data['RRab'].x_obs)),
    'RRc_G_points': int(len(optical_data['RRc'].x_obs)),
    'RRab_W2_points': int(len(infrared_data['RRab'].x_obs)),
    'RRc_W2_points': int(len(infrared_data['RRc'].x_obs)),
}

In [ ]:
derived_rows = []
for band_label, comparison_map in [('Gaia $G$', optical_data), ('WISE $W2$', infrared_data)]:
    for rr_class in ('RRab', 'RRc'):
        fit = comparison_map[rr_class]
        derived_rows.append(
            {
                'band': band_label,
                'class': rr_class,
                'N_fit': int(len(fit.x_obs)),
                'slope_q50': round(float(fit.slope_q50), 3),
                'slope_68pct': f"[{fit.slope_q16:.3f}, {fit.slope_q84:.3f}]",
                'sigma_scatter_q50': round(float(fit.sigma_scatter_q50), 3),
                'sigma_scatter_68pct': f"[{fit.sigma_scatter_q16:.3f}, {fit.sigma_scatter_q84:.3f}]",
            }
        )

derived_table = table.Table(rows=derived_rows)
derived_table

In [ ]:
literature_rows = [
    {
        "reference": "Klein et al. 2014 (inferred visual benchmark)",
        "band_or_scope": "Johnson V / classical optical context",
        "benchmark": "adopted a_V ≈ 0.0 mag dex^-1",
        "use_here": "Concrete numerical proxy for the paper statement that the optical RR Lyrae PL slope is negligible.",
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'Classical optical / visual context',
        'benchmark': 'Optical PL relation is negligible; infrared slopes steepen strongly.',
        'use_here': 'Treats Johnson V as a weak-period benchmark rather than a like-for-like Gaia G coefficient.',
    },
    {
        'reference': 'Beaton et al. 2018',
        'band_or_scope': 'Population II distance-indicator review',
        'benchmark': 'Calibration depends on bandpass, reddening, metallicity, and zero-point strategy.',
        'use_here': 'Explains why systematic offsets from our simple Gaia G fit are expected.',
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'WISE W2, RRab',
        'benchmark': 'a_W2 = -2.39 +/- 0.20',
        'use_here': 'Subclass-matched W2 slope benchmark for RRab.',
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'WISE W2, RRc',
        'benchmark': 'a_W2 = -1.70 +/- 0.62',
        'use_here': 'Subclass-matched W2 slope benchmark for RRc.',
    },
    {
        'reference': 'Dambis et al. 2014',
        'band_or_scope': 'WISE W2, mixed PLZ relation',
        'benchmark': '<M_W2> = gamma_W2 - (2.269 +/- 0.127) log P_F + (0.108 +/- 0.021) [Fe/H]',
        'use_here': 'Additional infrared reference showing that metallicity remains relevant even in W2.',
    },
]

literature_table = table.Table(rows=literature_rows)
literature_table

In [ ]:
rrab_g = optical_data['RRab']
rrc_g = optical_data['RRc']

optical_discussion = f"""
## Analysis: Gaia $G$ Versus the Classical Optical Literature

Our fitted Gaia $G$ relations are clearly non-zero: for RRab we find $a_G = {rrab_g.slope_q50:.3f}$ with a 68% interval of [{rrab_g.slope_q16:.3f}, {rrab_g.slope_q84:.3f}], and for RRc we find $a_G = {rrc_g.slope_q50:.3f}$ with a 68% interval of [{rrc_g.slope_q16:.3f}, {rrc_g.slope_q84:.3f}]. Those slopes are much stronger than the classical visual-band benchmark adopted here, $a_V \approx 0.0$ mag dex$^{-1}$, which I use as a numerical proxy for the [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) statement that the optical RR Lyrae PL slope is negligible and only becomes strongly constrained toward the infrared.

That difference is not automatically a contradiction. The first reason is bandpass. Gaia $G$ is not Johnson $V$; it is broader and reaches farther to the red, so it can naturally sit between the nearly-flat visual benchmark and the much steeper infrared behavior. The second reason is model specification. [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) review the fact that RR Lyrae calibration depends on wavelength, reddening, metallicity, and the way the absolute scale is established. Our upstream fit is intentionally simple: it is a period-luminosity model with intrinsic scatter, but no explicit metallicity term and no hierarchical treatment of the absolute scale. Any true PLZ dependence can therefore leak into both the fitted slope and the residual scatter.

The third reason is methodology. The upstream notebook works in direct absolute-magnitude space after converting Gaia parallaxes into $M_G$, whereas Gaia-era calibration papers generally recommend parallax-space or hierarchical Bayesian treatments because they better control inverse-parallax bias and zero-point systematics ([Luri et al. 2018](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html); [Muraveva et al. 2018](https://academic.oup.com/mnras/article/481/1/1195/5075596)). A fourth reason is sample construction: our Gaia sample is local, quality-cut, and explicitly split into RRab and RRc, while many published optical comparisons use different extinction corrections, metallicity baselines, or mixed/fundamentalized samples.

The defensible scientific conclusion is therefore modest. Our Gaia $G$ slopes should **not** be read as direct replacements for a Johnson $V$ PL coefficient. They should instead be read as evidence that a broad Gaia optical band already begins to pick up some of the stronger period dependence that becomes cleaner in the infrared. That interpretation is consistent with the multiwavelength picture in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) and the calibration caveats emphasized by [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1).
"""

display(Markdown(optical_discussion))

In [ ]:
comparison_rows = []
for rr_class in ('RRab', 'RRc'):
    g_fit = optical_data[rr_class]
    w2_fit = infrared_data[rr_class]
    klein_fit = literature['klein_w2'][rr_class]
    comparison_rows.append(
        {
            'class': rr_class,
            'G_slope': f"{g_fit.slope_q50:.3f} [{g_fit.slope_q16:.3f}, {g_fit.slope_q84:.3f}]",
            'W2_slope': f"{w2_fit.slope_q50:.3f} [{w2_fit.slope_q16:.3f}, {w2_fit.slope_q84:.3f}]",
            'W2_minus_G': round(float(w2_fit.slope_q50 - g_fit.slope_q50), 3),
            'Klein_W2_slope': f"{klein_fit['slope']:.2f} +/- {klein_fit['err']:.2f}",
            'W2_minus_Klein': round(float(w2_fit.slope_q50 - klein_fit['slope']), 3),
            'G_scatter': round(float(g_fit.sigma_scatter_q50), 3),
            'W2_scatter': round(float(w2_fit.sigma_scatter_q50), 3),
        }
    )

band_comparison_table = table.Table(rows=comparison_rows)
band_comparison_table

In [ ]:
rrab_w2 = infrared_data['RRab']
rrc_w2 = infrared_data['RRc']

infrared_discussion = f"""
## Analysis: $W2$ Versus $G$, and $W2$ Versus the Infrared Literature

The internal comparison is clear. For RRab, the slope changes from $a_G = {rrab_g.slope_q50:.3f}$ to $a_{{W2}} = {rrab_w2.slope_q50:.3f}$, while the intrinsic scatter drops from {rrab_g.sigma_scatter_q50:.3f} mag to {rrab_w2.sigma_scatter_q50:.3f} mag. For RRc, the slope changes from $a_G = {rrc_g.slope_q50:.3f}$ to $a_{{W2}} = {rrc_w2.slope_q50:.3f}$, and the intrinsic scatter drops from {rrc_g.sigma_scatter_q50:.3f} mag to {rrc_w2.sigma_scatter_q50:.3f} mag. That is exactly the direction expected from the physical argument in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776): once the bandpass moves into the infrared, reduced extinction sensitivity and a weaker temperature term make the relation steeper and tighter.

Relative to the published `W2` literature, however, our infrared slopes are still systematically steeper. For RRab, [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) report $a_{{W2}} = -2.39 ± 0.20$, compared to our {rrab_w2.slope_q50:.3f}. For RRc, they report $a_{{W2}} = -1.70 ± 0.62$, compared to our {rrc_w2.slope_q50:.3f}. [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) also find a somewhat shallower mixed-sample infrared slope, together with a non-negligible metallicity term in $W2$. That combination is informative: it suggests that the broad optical-to-infrared trend is reproduced, but the exact coefficient still depends on how the calibration is built.

Several effects can drive the remaining offset. First, the samples are genuinely different. Our `W2` fit is built from the Gaia-cleaned and Gaia+WISE-matched sample used in this lab, whereas the Klein et al. calibration used 104 RRab stars and only 19 RRc stars with prior distances tied to an established $M_V$-[Fe/H] scale plus a handful of HST parallaxes. Second, the literature often either fundamentalizes RRc periods or mixes subclasses in a PLZ framework, while our notebook keeps RRab and RRc fully separated. Third, even in the infrared, metallicity has not disappeared; [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) explicitly fit a metallicity term, while our notebook does not. Fourth, the absolute scale is still being set differently across analyses, and [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) stress that these zero-point and calibration choices remain a major systematic.

So the strongest conclusion is comparative rather than absolute. The notebook reproduces the key astrophysical trend: `W2` is steeper and tighter than Gaia $G$, consistent with the infrared literature. But the exact `W2` slopes are not identical to published calibrations, especially for RRc, and the likely explanation is a combination of bandpass, metallicity, subclass handling, sample construction, and calibration methodology rather than a single isolated failure of the fit.
"""

display(Markdown(infrared_discussion))

In [ ]:
# ============================================================
# 07-period-color.ipynb
# ============================================================

In [ ]:
from pathlib import Path
from types import SimpleNamespace

from astropy import table
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import (
    FULL_RRLYRAE_GAIA_SOURCE_QUERY,
    RRAB_RRC_GAIA_SOURCE_QUERY,
    build_optical_pc_comparison_data,
    build_pc_context,
    fit_pc_nuts,
    load_or_create_rrab_rrc_full_catalog,
    load_table_npz,
    mcmc_sampler_color,
    pc_posterior_summary_row,
    plot_corner,
    plot_pc_posterior_predictive,
    plot_pc_posterior_predictive_comparison,
    plot_trace,
    save_optical_pc_comparison_data,
)

In [ ]:
shared_catalog_path = Path("rrlyrae_rrab_rrc_full_catalog.npz")
rrlyrae_rrab_rrc_full, shared_catalog_status = load_or_create_rrab_rrc_full_catalog(shared_catalog_path)
shared_catalog_classes = np.asarray(rrlyrae_rrab_rrc_full["best_classification"]).astype(str)
shared_catalog_counts = table.Table(
    rows=[
        {"class": rr_class, "N": int(np.count_nonzero(shared_catalog_classes == rr_class))}
        for rr_class in ("RRab", "RRc")
    ]
)

print(
    f"Shared RRab/RRc catalog {shared_catalog_status}: "
    f"{len(rrlyrae_rrab_rrc_full):,} stars"
)
shared_catalog_counts


In [ ]:
data_path = Path('rrlyrae_calibration_sample.npz')
rrlyrae_clean_data = load_table_npz(data_path)

classes = np.asarray(rrlyrae_clean_data['best_classification']).astype(str)
class_labels, class_counts = np.unique(classes, return_counts=True)
class_count_table = table.Table({'class': class_labels, 'N': class_counts})

class_count_table


In [ ]:
scope_summary = table.Table(
    rows=[
        {
            "stage": "Handout item 24 query",
            "classes_in_scope": "All RR Lyrae classes",
            "used_downstream": "Query provenance only",
            "reason": "Shows the literal vari_rrlyrae + gaia_source cross-match requested in the handout.",
        },
        {
            "stage": "Saved shared cache",
            "classes_in_scope": "RRab, RRc",
            "used_downstream": "Yes",
            "reason": "Only subclasses with an intrinsic-color calibration derived in this lab.",
        },
        {
            "stage": "Excluded downstream classes",
            "classes_in_scope": "RRd and others",
            "used_downstream": "No",
            "reason": "No standalone period-color calibration is derived for those classes here.",
        },
    ]
)
scope_summary

In [ ]:
parallax_sensitivity_summary = table.Table(
    rows=[
        {
            "quantity": r"$(G_{BP}-G_{RP})_{obs}$",
            "d(quantity)/d(parallax)": 0.0,
            "include_parallax_in_sigma?": "No",
            "reason": "Apparent Gaia color is a difference of directly observed magnitudes and is distance-independent.",
        },
        {
            "quantity": r"$M_G$ or distance modulus",
            "d(quantity)/d(parallax)": "non-zero",
            "include_parallax_in_sigma?": "Not in this notebook",
            "reason": "Parallax enters distance-dependent quantities, but the period-color fit is to apparent color only.",
        },
    ]
)
parallax_sensitivity_summary

In [ ]:
DISPLAY_LABELS = [r"$a_c$", r"$b_c$", r"$\sigma_{c,\mathrm{scatter}}$"]
RR_CLASSES = ("RRab", "RRc")
FIT_SEED_MAP = {"RRab": 42, "RRc": 84}
FIT_STEPS = 10_000
FIT_BURN = 2_000

In [ ]:
rrab_data = rrlyrae_clean_data[classes == "RRab"]
rrc_data = rrlyrae_clean_data[classes == "RRc"]
rrd_data = rrlyrae_clean_data[classes == "RRd"]

pc_context = {
    "RRab": build_pc_context(rrab_data, "RRab"),
    "RRc": build_pc_context(rrc_data, "RRc"),
}

theta0_table = table.Table(
    rows=[
        {
            "class": rr_class,
            "N": pc_context[rr_class].n,
            "mean_log_period": round(float(pc_context[rr_class].x_mean), 4),
            "theta0_a_c": round(float(pc_context[rr_class].theta0[0]), 4),
            "theta0_b_c": round(float(pc_context[rr_class].theta0[1]), 4),
            "theta0_sigma_c": round(float(10.0 ** pc_context[rr_class].theta0[2]), 4),
        }
        for rr_class in RR_CLASSES
    ]
)

{"RRab": pc_context["RRab"].n, "RRc": pc_context["RRc"].n, "RRd excluded": len(rrd_data)}
theta0_table


In [ ]:
pc_results = {}
for rr_class in RR_CLASSES:
    fit = fit_pc_nuts(
        pc_context[rr_class],
        n_steps=FIT_STEPS,
        n_burn=FIT_BURN,
        seed=FIT_SEED_MAP[rr_class],
    )
    pc_results[rr_class] = {
        "ctx": pc_context[rr_class],
        "display": fit.display,
        "samples": fit.samples,
        "comparison_samples": fit.comparison_samples,
        "summary": pc_posterior_summary_row(rr_class, fit.samples, fit.acceptance_rate),
        "acceptance_rate": fit.acceptance_rate,
    }

pc_comparison_export = {
    rr_class: build_optical_pc_comparison_data(
        pc_results[rr_class]["ctx"],
        pc_results[rr_class]["comparison_samples"],
    )
    for rr_class in RR_CLASSES
}

summary_table = table.Table(
    rows=[pc_results[rr_class]["summary"] for rr_class in RR_CLASSES]
)

summary_table

In [ ]:
for rr_class in RR_CLASSES:
    display = pc_results[rr_class]["display"]
    axes = plot_trace(
        display.samples,
        display.log_probs,
        display.param_labels,
        display.n_burn,
        color=mcmc_sampler_color(rr_class, "native"),
    )
    axes[0].set_title(f"Trace: {rr_class} period-color relation (native PyMC NUTS)")
    plt.show()

In [ ]:
for rr_class in RR_CLASSES:
    fig = plot_corner(
        pc_results[rr_class]["display"],
        color=mcmc_sampler_color(rr_class, "native"),
    )
    fig.suptitle(f"Posterior: {rr_class} period-color relation (native PyMC NUTS)")
    plt.show()

In [ ]:
for rr_class in RR_CLASSES:
    save_name = {"RRab": "fig_pc_posterior_rrab.pdf", "RRc": "fig_pc_posterior_rrc.pdf"}[rr_class]
    ax = plot_pc_posterior_predictive(
        pc_results[rr_class]["ctx"],
        pc_results[rr_class]["samples"],
        save_name=save_name,
    )
    ax.set_title(f"{rr_class} posterior predictive check (native PyMC NUTS)")
    plt.show()

ax = plot_pc_posterior_predictive_comparison(
    pc_results["RRab"]["ctx"],
    pc_results["RRab"]["samples"],
    pc_results["RRc"]["ctx"],
    pc_results["RRc"]["samples"],
    save=True,
)
ax.set_title("RRab and RRc native-PyMC posterior predictive comparison")
plt.show()

In [ ]:
pc_output_path = Path('rrlyrae_optical_pc_comparison_data.npz')
save_optical_pc_comparison_data(pc_output_path, pc_comparison_export)

{
    'pc_artifact': str(pc_output_path.resolve()),
    'classes': list(pc_comparison_export),
    'shared_catalog_path': str(shared_catalog_path.resolve()),
    'shared_catalog_rows': len(rrlyrae_rrab_rrc_full),
}


In [ ]:
# ============================================================
# 08-extinction.ipynb
# ============================================================

In [ ]:
from pathlib import Path

import numpy as np
from astropy import table

from ugdatalab import (
    build_reddening_quality_mask,
    compute_period_color_extinction,
    empirical_vs_catalog_extinction,
    load_optical_pc_comparison_data,
    load_table_npz,
    plot_empirical_vs_catalog_extinction_comparison,
    rrlyrae_class_mask,
    save_table_npz,
)

PC_COMPARISON_PATH = Path("rrlyrae_optical_pc_comparison_data.npz")
FULL_CATALOG_PATH = Path("rrlyrae_rrab_rrc_full_catalog.npz")
EXTINCTION_CATALOG_PATH = Path("rrlyrae_extinction_catalog.npz")
R_G = 2.0

In [ ]:
try:
    pc_comparison = load_optical_pc_comparison_data(PC_COMPARISON_PATH)
    rrlyrae_full = load_table_npz(FULL_CATALOG_PATH)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Required local .npz handoff files were not available in labs/01. '
        'Run 05.ipynb to regenerate them before executing this notebook.'
    ) from exc

pc_summaries = {
    rr_class: {
        'slope_median': comparison.slope_median,
        'slope_std': comparison.slope_std,
        'intercept_median': comparison.intercept_median,
        'intercept_std': comparison.intercept_std,
        'intrinsic_sigma_median': comparison.intrinsic_sigma_median,
        'intrinsic_sigma_std': comparison.intrinsic_sigma_std,
    }
    for rr_class, comparison in pc_comparison.items()
}

class_counts = {
    'RRab': int(rrlyrae_class_mask(rrlyrae_full, 'RRab').sum()),
    'RRc': int(rrlyrae_class_mask(rrlyrae_full, 'RRc').sum()),
}

print(f"Shared RRab/RRc catalog: {len(rrlyrae_full):,} stars")
for rr_class, count in class_counts.items():
    print(f"  {rr_class}: {count:,}")

posterior_summary_table = table.Table(
    rows=[
        {
            'class': rr_class,
            'n_fit': int(len(comparison.x_obs)),
            'slope_median': round(comparison.slope_median, 4),
            'slope_std': round(comparison.slope_std, 4),
            'intercept_median': round(comparison.intercept_median, 4),
            'intercept_std': round(comparison.intercept_std, 4),
            'sigma_c_median': round(comparison.intrinsic_sigma_median, 4),
        }
        for rr_class, comparison in pc_comparison.items()
    ]
)
posterior_summary_table


In [ ]:
extinction_scope_table = table.Table(
    rows=[
        {
            "artifact": "rrlyrae_rrab_rrc_full_catalog.npz",
            "classes_present": "RRab, RRc",
            "rows": int(len(rrlyrae_full)),
            "empirical_A_G_defined": "Yes",
            "note": "These are the subclasses with calibrated intrinsic-color relations from 05.ipynb.",
        },
        {
            "artifact": "Full Gaia RR Lyrae catalog outside this cache",
            "classes_present": "RRd and other classes not loaded here",
            "rows": "not carried into 06.ipynb",
            "empirical_A_G_defined": "No",
            "note": "The lab does not derive an intrinsic-color calibration for those classes.",
        },
    ]
)
extinction_scope_table

In [ ]:
rrlyrae_extinction = compute_period_color_extinction(
    rrlyrae_full,
    pc_summaries,
    r_g=R_G,
    copy=True,
)

valid_empirical = np.isfinite(np.asarray(rrlyrae_extinction['E_bprp'], dtype=float))
valid_by_class = table.Table(
    rows=[
        {
            'class': rr_class,
            'valid_empirical_rows': int((valid_empirical & rrlyrae_class_mask(rrlyrae_extinction, rr_class)).sum()),
        }
        for rr_class in ('RRab', 'RRc')
    ]
)
valid_by_class


In [ ]:
save_table_npz(EXTINCTION_CATALOG_PATH, rrlyrae_extinction)
{"saved": str(EXTINCTION_CATALOG_PATH.resolve()), "rows": len(rrlyrae_extinction)}

In [ ]:
from IPython.display import display

empirical = np.asarray(rrlyrae_extinction['A_G_calc'], dtype=float)
catalog = np.asarray(rrlyrae_extinction['g_absorption'], dtype=float)
phot_g = np.asarray(rrlyrae_extinction['phot_g_mean_mag'], dtype=float)

empirical_finite_mask = np.isfinite(empirical)
catalog_negative_mask = np.isfinite(catalog) & (catalog < 0.0)
catalog_too_large_mask = np.isfinite(catalog) & np.isfinite(phot_g) & (catalog > phot_g + 1.0)
catalog_outlier_mask = catalog_negative_mask | catalog_too_large_mask
catalog_physical_mask = np.isfinite(catalog) & ~catalog_outlier_mask
reddening_quality_mask = build_reddening_quality_mask(rrlyrae_extinction)
comparison_mask = empirical_finite_mask & catalog_physical_mask & reddening_quality_mask

catalog_outlier_reason = np.full(len(rrlyrae_extinction), '', dtype='U20')
catalog_outlier_reason[catalog_negative_mask] = 'negative_A_G'
catalog_outlier_reason[catalog_too_large_mask] = 'A_G_gt_G_plus_1'
rrlyrae_extinction['catalog_ag_is_physical'] = catalog_physical_mask
rrlyrae_extinction['catalog_ag_outlier_reason'] = catalog_outlier_reason
rrlyrae_extinction['reddening_quality_ok'] = reddening_quality_mask
rrlyrae_extinction['extinction_comparison_ok'] = comparison_mask

sample_masks = [
    ('All cached rows', np.ones(len(rrlyrae_extinction), dtype=bool)),
    ('RRab', rrlyrae_class_mask(rrlyrae_extinction, 'RRab')),
    ('RRc', rrlyrae_class_mask(rrlyrae_extinction, 'RRc')),
]

def _stage_summary(label, mask):
    return {
        'sample': label,
        'total_rows': int(mask.sum()),
        'empirical_rows': int((mask & empirical_finite_mask).sum()),
        'finite_catalog_rows': int((mask & np.isfinite(catalog)).sum()),
        'physical_catalog_rows': int((mask & catalog_physical_mask).sum()),
        'quality_rows': int((mask & reddening_quality_mask).sum()),
        'comparison_rows': int((mask & comparison_mask).sum()),
    }

sample_flow_summary = table.Table(
    rows=[_stage_summary(label, mask) for label, mask in sample_masks]
)

def _mask_summary(mask, label):
    values = catalog[mask]
    return {
        'criterion': label,
        'N': int(mask.sum()),
        'min_g_absorption': np.nan if len(values) == 0 else round(float(np.min(values)), 4),
        'max_g_absorption': np.nan if len(values) == 0 else round(float(np.max(values)), 4),
    }

catalog_outlier_summary = table.Table(
    rows=[
        _mask_summary(catalog_negative_mask, 'g_absorption < 0'),
        _mask_summary(catalog_too_large_mask, 'g_absorption > G + 1'),
    ]
)

catalog_outliers = rrlyrae_extinction[catalog_outlier_mask][
    'source_id',
    'best_classification',
    'phot_g_mean_mag',
    'g_absorption',
    'g_absorption_error',
    'bp_rp',
    'l',
    'b',
    'catalog_ag_outlier_reason',
].copy()
catalog_outliers.sort('g_absorption')

positive_catalog_outliers = catalog_outliers[
    catalog_outliers['catalog_ag_outlier_reason'] == 'A_G_gt_G_plus_1'
]

positive_catalog_outliers = positive_catalog_outliers[
    'source_id',
    'best_classification',
    'phot_g_mean_mag',
    'g_absorption',
    'g_absorption_error',
    'bp_rp',
    'l',
    'b',
].copy()

comparison = empirical_vs_catalog_extinction(rrlyrae_extinction[comparison_mask])

def _residual_summary(mask, label):
    finite = comparison_mask & mask
    residual = empirical[finite] - catalog[finite]
    if len(residual) == 0:
        return {
            'sample': label,
            'N': 0,
            'median_resid': np.nan,
            'mad_sigma': np.nan,
            'rms_resid': np.nan,
        }
    med = float(np.median(residual))
    mad_sigma = float(1.4826 * np.median(np.abs(residual - med)))
    rms = float(np.sqrt(np.mean(residual**2)))
    return {
        'sample': label,
        'N': int(finite.sum()),
        'median_resid': round(med, 4),
        'mad_sigma': round(mad_sigma, 4),
        'rms_resid': round(rms, 4),
    }

comparison_summary = table.Table(
    rows=[
        _residual_summary(np.ones(len(rrlyrae_extinction), dtype=bool), 'Quality-filtered matches'),
        _residual_summary(rrlyrae_class_mask(rrlyrae_extinction, 'RRab'), 'RRab'),
        _residual_summary(rrlyrae_class_mask(rrlyrae_extinction, 'RRc'), 'RRc'),
    ]
)

display(sample_flow_summary)
display(catalog_outlier_summary)
comparison_summary


In [ ]:
fig, axes = plot_empirical_vs_catalog_extinction_comparison(
    comparison.catalog,
    comparison.empirical,
    comparison.residuals,
    save=True,
)


In [ ]:
from IPython.display import Markdown, display

all_row = comparison_summary[0]
rrab_row = comparison_summary[1]
rrc_row = comparison_summary[2]

discussion = rf"""
## Analysis and Discussion

This comparison should be interpreted as a test of consistency between two different extinction constructions, not as a demand for exact equality. Our empirical $A_G^{{calc}}$ is inferred from observed Gaia color minus a class-specific intrinsic RR Lyrae period-color relation, while Gaia DR3 `g_absorption` is tied to the extinction treatment used in the RR Lyrae pipeline and the underlying SFD dust-map framework (Clementini et al. 2023; Schlegel, Finkbeiner, & Davis 1998; Schlafly & Finkbeiner 2011).

In the current quality-filtered comparison sample, the residual $A_G^{{calc}} - g_{{absorption}}$ has median **{all_row['median_resid']:.4f} mag**, MAD-based scatter **{all_row['mad_sigma']:.4f} mag**, and RMS **{all_row['rms_resid']:.4f} mag** across **{all_row['N']}** matched stars. That is not star-by-star equality, but it is good enough to show that the empirical period-color calibration is recovering the same broad extinction scale as the Gaia RR Lyrae pipeline for the subset where both quantities are defined. The median offset is modest compared with the full residual width, so the main limitation is scatter rather than a catastrophic global bias.

The comparison is effectively **RRab-only**. The RRab row gives median residual **{rrab_row['median_resid']:.4f} mag** with RMS **{rrab_row['rms_resid']:.4f} mag**, while the RRc comparison row has **N = {rrc_row['N']}** because the current cached artifact contains no finite RRc `g_absorption` values even though empirical extinction quantities are still computed for RRc. That RRab-only behavior is consistent with the Gaia DR3 RR Lyrae release itself. Clementini et al. (2023) report that Gaia published an interstellar absorption estimate for 142 660 fundamental-mode RR Lyrae stars, derived from a relation involving the $G$-band amplitude, the $(G-G_{{\mathrm{{RP}}}})$ color, and the pulsation period. They do not present the same product as a general RRc extinction column.

After applying both the catalog sanity cut and the reddening-quality mask, the remaining largest empirical-minus-catalog differences should still be read cautiously. They can reflect imperfections in the class-specific intrinsic-color model, limits of a fixed $R_G = 2.0$ conversion, or differences between a star-by-star empirical estimate and a dust-map-based catalog value that effectively traces the line-of-sight dust column rather than the exact extinction to that pulsator. In other words, disagreement does not automatically imply one column is wrong; it often indicates that the two estimates are sensitive to different pieces of the extinction problem. The raw unfiltered scatter should therefore not be overinterpreted as a failure of the period-color relation by itself.

This notebook is intentionally scoped to RRab and RRc because the shared full-catalog cache and the saved period-color fit artifact are both defined only for the two single-mode subclasses. RRd is therefore excluded upstream rather than being carried through with undefined empirical-extinction fields.

### References

- Beaton, R. L., Bono, G., Braga, V. F., et al. (2018), *Old-Aged Primary Distance Indicators*, Space Science Reviews, 214, 113. https://doi.org/10.1007/s11214-018-0542-1
- Bono, G., Iannicola, G., Braga, V. F., et al. (2019), *On a New Method to Estimate the Distance, Reddening, and Metallicity of RR Lyrae Stars Using Optical/Near-infrared (B, V, I, J, H, K) Mean Magnitudes: $\omega$ Centauri as a First Test Case*, ApJ, 870, 115. https://doi.org/10.3847/1538-4357/aaf23f
- Clementini, G., Ripepi, V., Garofalo, A., et al. (2023), *Gaia Data Release 3: Specific processing and validation of all-sky RR Lyrae and Cepheid stars. The RR Lyrae sample*, A&A, 674, A18. https://doi.org/10.1051/0004-6361/202243964
- Schlegel, D. J., Finkbeiner, D. P., & Davis, M. (1998), *Maps of Dust Infrared Emission for Use in Estimation of Reddening and Cosmic Microwave Background Radiation Foregrounds*, ApJ, 500, 525. https://doi.org/10.1086/305772
- Schlafly, E. F., & Finkbeiner, D. P. (2011), *Measuring Reddening with Sloan Digital Sky Survey Stellar Spectra and Recalibrating SFD*, ApJ, 737, 103. https://doi.org/10.1088/0004-637X/737/2/103
"""

display(Markdown(discussion))

In [ ]:
# ============================================================
# 09-reddening-map-sfd.ipynb
# ============================================================

In [ ]:

from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
from astropy import table

from ugdatalab import (
    build_quality_components,
    build_stage_summary,
    build_criterion_failure_table,
    build_reddening_quality_mask,
    load_table_npz,
    plot_aitoff_reddening_map,
    plot_quality_diagnostics,
    rrlyrae_class_mask,
    plot_reddening_distribution,
)

PC_COMPARISON_PATH = Path("rrlyrae_optical_pc_comparison_data.npz")
FULL_CATALOG_PATH = Path("rrlyrae_rrab_rrc_full_catalog.npz")
EXTINCTION_CATALOG_PATH = Path("rrlyrae_extinction_catalog.npz")
R_G = 2.0

RR_CLASSES = ("RRab", "RRc")
MIN_BP_SNR = 5.0
MIN_RP_SNR = 5.0
APPLY_BP_RP_EXCESS_CUT = True
MAX_SIGMA_E = 0.15
MIN_EBPRP = 0.0           # physically motivated: dust only reddens
MAX_EBPRP = 10.0          # physical upper ceiling (mag, Gaia BP-RP)
MIN_REDDENING_SNR = None  # set to float (e.g. 1.0) to enable
MIN_RETAINED_STARS = 60_000
DIAGNOSTIC_SAMPLE_SIZE = 25_000
RNG_SEED = 7


In [ ]:

try:
    rrlyrae_extinction = load_table_npz(EXTINCTION_CATALOG_PATH)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Required artifact rrlyrae_extinction_catalog.npz not found in labs/01. '
        'Run 06.ipynb to regenerate it before executing this notebook.'
    ) from exc

finite_empirical = np.isfinite(np.asarray(rrlyrae_extinction['E_bprp'], dtype=float))
full_catalog_summary = table.Table(
    rows=[
        {
            'class': rr_class,
            'N_catalog': int(rrlyrae_class_mask(rrlyrae_extinction, rr_class).sum()),
            'N_finite_E': int((finite_empirical & rrlyrae_class_mask(rrlyrae_extinction, rr_class)).sum()),
        }
        for rr_class in RR_CLASSES
    ]
)

print(f"Shared RRab/RRc catalog: {len(rrlyrae_extinction):,} stars")
display(full_catalog_summary)

In [ ]:

components = build_quality_components(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    max_sigma_e=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
)

adopted_mask = build_reddening_quality_mask(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    apply_bp_rp_excess_cut=APPLY_BP_RP_EXCESS_CUT,
    max_sigma_E=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
    min_reddening_snr=MIN_REDDENING_SNR,
)
if not np.array_equal(adopted_mask, components['adopted']):
    raise AssertionError('Notebook-local mask components disagreed with build_reddening_quality_mask().')

removed_mask = components['finite'] & ~adopted_mask
e_valid = np.asarray(rrlyrae_extinction['E_bprp'], dtype=float)[components['finite']]
color_vmin = float(min(0.0, np.nanpercentile(e_valid, 0.5)))
color_vmax = float(np.nanpercentile(e_valid, 99.5))

stage_summary = build_stage_summary(rrlyrae_extinction, components)
criterion_failures = build_criterion_failure_table(rrlyrae_extinction, components)
final_summary = table.Table(
    rows=[
        {
            'N_retained': int(adopted_mask.sum()),
            'N_removed': int(removed_mask.sum()),
            'RRab_retained': int(np.count_nonzero(adopted_mask & rrlyrae_class_mask(rrlyrae_extinction, 'RRab'))),
            'RRc_retained': int(np.count_nonzero(adopted_mask & rrlyrae_class_mask(rrlyrae_extinction, 'RRc'))),
            'fraction_of_full': round(float(adopted_mask.sum()) / len(rrlyrae_extinction), 4),
            'fraction_of_finite': round(float(adopted_mask.sum()) / int(components['finite'].sum()), 4),
        }
    ]
)

if int(adopted_mask.sum()) < MIN_RETAINED_STARS:
    raise RuntimeError(
        f'Adopted cut retains only {int(adopted_mask.sum()):,} stars, below the required {MIN_RETAINED_STARS:,}.'
    )

display(stage_summary)
display(criterion_failures)
display(final_summary)
print(f'Adopted quality mask retains {int(adopted_mask.sum()):,} stars.')


In [ ]:

fig_uncut, ax_uncut = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    components['finite'],
    title='Uncut RR Lyrae reddening map',
    vmin=color_vmin,
    vmax=color_vmax,
)
plt.show()


In [ ]:

fig_diag, axes_diag = plot_quality_diagnostics(
    rrlyrae_extinction,
    components,
    sample_size=DIAGNOSTIC_SAMPLE_SIZE,
    seed=RNG_SEED,
    save=True,
)
plt.show()


In [ ]:
# Physical reddening value diagnostics — applied to the stage-5 (sigma_E-passed) sample
stage5_mask = (
    components['finite']
    & components['bp_snr']
    & components['rp_snr']
    & components['bp_rp_excess']
    & components['sigma_e']
)
e_stage5 = np.asarray(rrlyrae_extinction['E_bprp'], dtype=float)[stage5_mask]

n_negative = int((e_stage5 < 0).sum())
n_extreme = int((e_stage5 > MAX_EBPRP).sum())
print(f"Stars with E(BP-RP) < 0 in stage-5 sample:        {n_negative:,}  ({100*n_negative/len(e_stage5):.2f}%)")
print(f"Stars with E(BP-RP) > {MAX_EBPRP:.1f} in stage-5 sample:  {n_extreme:,}  ({100*n_extreme/len(e_stage5):.2f}%)")
print(f"Stage-5 sample size:                               {len(e_stage5):,}")

fig_phys, ax_phys = plot_reddening_distribution(e_stage5, min_ebprp=MIN_EBPRP, max_ebprp=MAX_EBPRP, save=True)
plt.show()


In [ ]:

fig_cut, ax_cut = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    adopted_mask,
    title='Quality-cut RR Lyrae reddening map',
    vmin=color_vmin,
    vmax=color_vmax,
    save="fig_reddening_map.pdf",
)
plt.show()


In [ ]:

fig_removed, ax_removed = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    removed_mask,
    title='Stars removed by the adopted quality mask',
    vmin=color_vmin,
    vmax=color_vmax,
    alpha=0.28,
    size=3.0,
    save_name="fig_removed_stars_map.pdf",
)
plt.show()


In [ ]:
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
from astropy import table

from ugdatalab import (
    _cache_stable,
    binned_median_trend,
    build_quality_components,
    build_reddening_quality_mask,
    load_table_npz,
    plot_aitoff_sfd_map,
    plot_aitoff_value_map,
    plot_reddening_distribution,
    plot_regime_decomposition,
    plot_sfd_all_sky_hexbin,
    plot_sfd_empirical_hexbin_comparison,
    rank_spearman,
    rrlyrae_class_mask,
    sample_sfd_ebv,
    subset_row,
)

PC_COMPARISON_PATH = Path("rrlyrae_optical_pc_comparison_data.npz")
FULL_CATALOG_PATH = Path("rrlyrae_rrab_rrc_full_catalog.npz")
EXTINCTION_CATALOG_PATH = Path("rrlyrae_extinction_catalog.npz")
DUSTMAPS_DATA_DIR = Path('.dustmaps-data')
R_G = 2.0

RR_CLASSES = ("RRab", "RRc")
MIN_BP_SNR = 5.0
MIN_RP_SNR = 5.0
MAX_SIGMA_E = 0.15
MIN_EBPRP = 0.0           # physically motivated: dust only reddens
MAX_EBPRP = 10.0          # physical upper ceiling (mag, Gaia BP-RP)
MIN_REDDENING_SNR = None  # set to float (e.g. 1.0) to enable
MIN_RETAINED_STARS = 60_000
PLANE_LATITUDE_MAX = 15.0
HIGH_LATITUDE_MIN = 30.0
BIN_COUNT = 18


In [ ]:

try:
    rrlyrae_extinction = load_table_npz(EXTINCTION_CATALOG_PATH)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Required artifact rrlyrae_extinction_catalog.npz not found in labs/01. '
        'Run 06.ipynb to regenerate it before executing this notebook.'
    ) from exc

adopted_mask = build_reddening_quality_mask(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    apply_bp_rp_excess_cut=True,
    max_sigma_E=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
    min_reddening_snr=MIN_REDDENING_SNR,
)

if int(adopted_mask.sum()) < MIN_RETAINED_STARS:
    raise RuntimeError(
        f'Adopted cut retains only {int(adopted_mask.sum()):,} stars, below the required {MIN_RETAINED_STARS:,}.'
    )

rrlyrae_clean = rrlyrae_extinction[adopted_mask].copy()
cleaned_summary = table.Table(
    rows=[
        {
            'class': rr_class,
            'N_retained': int(np.count_nonzero(rrlyrae_class_mask(rrlyrae_clean, rr_class))),
        }
        for rr_class in RR_CLASSES
    ]
)

print(f'Quality-cut RR Lyrae sample: {len(rrlyrae_clean):,} stars')
display(cleaned_summary)

In [ ]:

try:
    from dustmaps.config import config as dustmaps_config
    import dustmaps.sfd
    from dustmaps.sfd import SFDQuery
except ImportError as exc:
    raise RuntimeError(
        'dustmaps is required for the SFD comparison notebook. '
        'Install the optional dust dependency and rerun this notebook.'
    ) from exc

DUSTMAPS_DATA_DIR.mkdir(parents=True, exist_ok=True)
dustmaps_config['data_dir'] = str(DUSTMAPS_DATA_DIR.resolve())


def ensure_sfd_available():
    try:
        SFDQuery()
        return 'available'
    except FileNotFoundError:
        print('SFD files were not found locally; downloading the SFD map...')
        try:
            dustmaps.sfd.fetch()
        except Exception as exc:
            raise RuntimeError(
                'SFD files were missing and the notebook could not download them. '
                'Check network access, rerun this cell, and then continue.'
            ) from exc
        SFDQuery()
        return 'downloaded'


@_cache_stable(module='ugdatalab.dust')
def sample_sfd_cached(l_deg, b_deg):
    coords = table.Table(
        {
            'l': np.asarray(l_deg, dtype=float),
            'b': np.asarray(b_deg, dtype=float),
        }
    )
    return sample_sfd_ebv(coords)


sfd_status = ensure_sfd_available()
rrlyrae_clean['sfd_ebv'] = sample_sfd_cached(
    np.asarray(rrlyrae_clean['l'], dtype=float),
    np.asarray(rrlyrae_clean['b'], dtype=float),
)

sfd_finite_mask = np.isfinite(np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float))
if not np.all(sfd_finite_mask):
    rrlyrae_clean = rrlyrae_clean[sfd_finite_mask].copy()

sfd_summary = table.Table(
    rows=[
        {
            'N_retained_rrlyrae': len(rrlyrae_clean),
            'RRab_retained': int(np.count_nonzero(rrlyrae_class_mask(rrlyrae_clean, 'RRab'))),
            'RRc_retained': int(np.count_nonzero(rrlyrae_class_mask(rrlyrae_clean, 'RRc'))),
            'finite_sfd_rows': int(np.count_nonzero(sfd_finite_mask)),
            'sfd_status': sfd_status,
        }
    ]
)
display(sfd_summary)


In [ ]:

# SFD E(B-V) value diagnostics
sfd_arr = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)
print(f"Negative SFD E(B-V): {int((sfd_arr < 0).sum())} stars")
print(f"SFD E(B-V) > 5.0:    {int((sfd_arr > 5.0).sum())} stars")
print(f"Median SFD E(B-V):   {np.nanmedian(sfd_arr):.4f} mag")
print(f"Total stars with SFD values: {int(np.isfinite(sfd_arr).sum())}")


In [ ]:

empirical = np.asarray(rrlyrae_clean['E_bprp'], dtype=float)
sfd_ebv = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)
l_deg = np.asarray(rrlyrae_clean['l'], dtype=float)
b_deg = np.asarray(rrlyrae_clean['b'], dtype=float)

empirical_vmin = float(min(0.0, np.nanpercentile(empirical, 0.5)))
empirical_vmax = float(np.nanpercentile(empirical, 99.5))
sfd_vmin = float(max(0.0, np.nanpercentile(sfd_ebv, 0.5)))
sfd_vmax = float(np.nanpercentile(sfd_ebv, 99.5))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=180, subplot_kw={'projection': 'aitoff'})
sc_empirical, label_empirical = plot_aitoff_value_map(
    axes[0],
    l_deg,
    b_deg,
    empirical,
    title='RR Lyrae empirical $E(G_{\\mathrm{BP}}-G_{\\mathrm{RP}})$',
    vmin=empirical_vmin,
    vmax=empirical_vmax,
    colorbar_label=r'$E(G_{\mathrm{BP}} - G_{\mathrm{RP}})$ [mag]',
)
sc_sfd, label_sfd = plot_aitoff_value_map(
    axes[1],
    l_deg,
    b_deg,
    sfd_ebv,
    title='Sampled SFD $E(B-V)$',
    vmin=sfd_vmin,
    vmax=sfd_vmax,
    colorbar_label=r'SFD $E(B-V)$ [mag]',
    cmap='viridis',
)

fig.colorbar(sc_empirical, ax=axes[0], orientation='horizontal', pad=0.08, shrink=0.9, label=label_empirical)
fig.colorbar(sc_sfd, ax=axes[1], orientation='horizontal', pad=0.08, shrink=0.9, label=label_sfd)
fig.suptitle(f'Matched-sightline comparison on {len(rrlyrae_clean):,} cleaned RR Lyrae stars', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
sfd_vmin = float(np.nanpercentile(sfd_ebv, 0.5))
sfd_vmax = float(np.nanpercentile(sfd_ebv, 99.5))

fig_sfd_map, ax_sfd_map = plot_aitoff_sfd_map(
    rrlyrae_clean,
    all_mask,
    title=r'SFD $E(B-V)$ sampled at RR Lyrae positions',
    vmin=sfd_vmin,
    vmax=sfd_vmax,
    save='fig_sfd_map.pdf',
)
plt.show()

In [ ]:
abs_b = np.abs(np.asarray(rrlyrae_clean['b'], dtype=float))
all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
plane_mask = abs_b < PLANE_LATITUDE_MAX
intermediate_mask = (abs_b >= PLANE_LATITUDE_MAX) & (abs_b < HIGH_LATITUDE_MIN)
high_lat_mask = abs_b >= HIGH_LATITUDE_MIN

comparison_summary = table.Table(
    rows=[
        subset_row('all cleaned stars', rrlyrae_clean, all_mask),
        subset_row(r'|b| < 15 deg', rrlyrae_clean, plane_mask),
        subset_row(r'15 <= |b| < 30 deg', rrlyrae_clean, intermediate_mask),
        subset_row(r'|b| >= 30 deg', rrlyrae_clean, high_lat_mask),
    ]
)
display(comparison_summary)

fig, ax = plot_sfd_all_sky_hexbin(rrlyrae_clean, save='fig_sfd_comparison.pdf')
plt.show()

In [ ]:
# Latitude-binned Pearson R² table
from scipy.stats import pearsonr

abs_b_all = np.abs(np.asarray(rrlyrae_clean['b'], dtype=float))
empirical_all = np.asarray(rrlyrae_clean['E_bprp'], dtype=float)
sfd_all = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)

bin_edges = np.arange(0, 91, 10)  # 0-10, 10-20, ..., 80-90
lat_rows = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    sel = (abs_b_all >= lo) & (abs_b_all < hi)
    n = int(sel.sum())
    if n >= 10:
        r, pval = pearsonr(sfd_all[sel], empirical_all[sel])
        r2 = r**2
    else:
        r2, pval = float('nan'), float('nan')
    lat_rows.append({
        '|b| bin': f'{lo:.0f}–{hi:.0f}°',
        'N_stars': n,
        'R2': round(float(r2), 4),
        'p_value': f'{pval:.2e}' if not np.isnan(pval) else 'N/A',
    })

lat_table = table.Table(rows=lat_rows)
print("Latitude-binned Pearson R² between SFD E(B-V) and RR Lyrae E(BP-RP):")
display(lat_table)

In [ ]:
SIMILAR_SCALE_MAX = 2.0   # E(B-V) [mag]: upper edge of "comparable" regime
LARGE_SFD_MIN = 10.0      # E(B-V) [mag]: lower edge of "large SFD" regime

finite_mask = np.isfinite(sfd_ebv) & np.isfinite(empirical)
similar_mask = finite_mask & (sfd_ebv <= SIMILAR_SCALE_MAX)
large_mask   = finite_mask & (sfd_ebv >  LARGE_SFD_MIN)

print(f"Similar-scale regime (SFD ≤ {SIMILAR_SCALE_MAX} mag): {similar_mask.sum():,} stars")
print(f"Large-SFD regime     (SFD > {LARGE_SFD_MIN} mag):  {large_mask.sum():,} stars")
print(f"Transition (2 < SFD ≤ 10 mag):  {(finite_mask & (sfd_ebv > SIMILAR_SCALE_MAX) & (sfd_ebv <= LARGE_SFD_MIN)).sum():,} stars")

In [ ]:
# --- linear fit in the similar-scale regime ---
from scipy.stats import pearsonr

x_sim = sfd_ebv[similar_mask]
y_sim = empirical[similar_mask]
slope, intercept = np.polyfit(x_sim, y_sim, 1)
r_sim, _ = pearsonr(x_sim, y_sim)
r2_sim = r_sim**2

fig, axes = plot_regime_decomposition(
    sfd_ebv, empirical,
    similar_mask=similar_mask,
    large_mask=large_mask,
    slope=slope,
    intercept=intercept,
    r2_sim=r2_sim,
    similar_scale_max=SIMILAR_SCALE_MAX,
    large_sfd_min=LARGE_SFD_MIN,
    save='fig_regime_decomposition.pdf',
)
plt.show()

In [ ]:
from IPython.display import Markdown

similar_median_empirical = float(np.nanmedian(empirical[similar_mask]))
similar_median_sfd       = float(np.nanmedian(sfd_ebv[similar_mask]))
large_median_empirical   = float(np.nanmedian(empirical[large_mask])) if large_mask.sum() > 0 else float('nan')
large_median_sfd         = float(np.nanmedian(sfd_ebv[large_mask]))  if large_mask.sum() > 0 else float('nan')

regime_discussion = rf"""
### Similar-Scale Regime (SFD $\leq$ {SIMILAR_SCALE_MAX} mag)

In the {similar_mask.sum():,}-star similar-scale sample (median SFD = {similar_median_sfd:.3f} mag, median empirical = {similar_median_empirical:.3f} mag), the two dust tracers are in the same dynamic range and the Pearson $R^2 = {r2_sim:.3f}$. The linear fit slope is **{slope:.3f}**, close to the expected Gaia-band conversion factor $R_{{BP-RP}} = E(G_{{BP}}-G_{{RP}})/E(B-V) \approx 1.3$ for a standard $R_V = 3.1$ extinction law ([Fitzpatrick 1999](https://doi.org/10.1086/316293); [Wang & Chen 2019](https://doi.org/10.3847/1538-4357/ab15f2)). Agreement at this level confirms that the empirical period-color calibration is recovering the same physical reddening as SFD in low-to-moderate dust columns.

### Large-SFD Regime (SFD $>$ {LARGE_SFD_MIN} mag)

The {large_mask.sum()} stars in the large-SFD sample (median SFD = {large_median_sfd:.1f} mag) show a striking decoupling: the empirical E(G_{{BP}}-G_{{RP}}) remains near {large_median_empirical:.2f} mag while SFD values span the full range above 10 mag. Two physical effects combine to produce this apparent saturation:

1. **Distance-limited sampling.** RR Lyrae are point sources at fixed distances (~1–30 kpc). The SFD map integrates dust all the way to infinity. In the heavily obscured galactic plane, the bulk of the dust column lies beyond the star, so the stellar measurement records only a fraction of the SFD column.

2. **Gaia photometric quality cuts.** Stars in highly obscured sightlines tend to have poor BP/RP flux ratios (low SNR, crowded fields), and they fail the `phot_bp/rp_mean_flux_over_error > 5` criterion applied in nb07. The surviving sample is already biased toward foreground, moderately reddened stars.

The large-SFD regime therefore does not indicate calibration failure: it illustrates exactly the regime where a 2D total-column map (SFD) and a 3D distance-limited stellar measurement (this work) are expected to diverge, and where a true 3D dust map such as Bayestar ([Green et al. 2019](https://doi.org/10.3847/1538-4357/ab5362)) is required.
"""

display(Markdown(regime_discussion))

In [ ]:
from IPython.display import Markdown, display

r2_0_10  = lat_table['R2'][0]
r2_10_20 = lat_table['R2'][1]
r2_20_30 = lat_table['R2'][2]
r2_30_40 = lat_table['R2'][3]

discussion = rf"""
## Analysis and Discussion

The side-by-side maps show that the broad Galactic structure agrees well. Both maps become redder toward the Galactic plane and the inner Milky Way, which is exactly what one expects if both are tracing the large-scale dust distribution ([Schlegel, Finkbeiner, & Davis 1998](https://doi.org/10.1086/305772); [Schlafly & Finkbeiner 2011](https://doi.org/10.1088/0004-637X/737/2/103)). The matched-sightline density plot should therefore be read mainly as a monotonic trend check rather than as a one-to-one calibration.

### Physical Interpretation of 2D vs. 3D

SFD is a two-dimensional, total-column reddening map calibrated from IRAS and DIRBE far-infrared emission with a temperature correction derived from the 100 μm / 60 μm flux ratio ([Schlegel, Finkbeiner, & Davis 1998](https://doi.org/10.1086/305772)). By construction it integrates the full dust column from here to infinity, providing a single number per sightline independent of distance. Our RR Lyrae map, by contrast, measures the color excess only out to the finite distance of each individual pulsator, which traces the old stellar population at distances of order 1–30 kpc.

At high Galactic latitudes ($|b| \gtrsim 30°$) the thin-disk dust scale height ($\\approx 150$–200 pc) means that most of the SFD column is accumulated within the first few hundred parsecs of the disk, well inside the typical RR Lyrae distance. In that regime the two measurements should agree well in a statistical sense. Near the plane ($|b| \lesssim 15°$) the agreement degrades in physical interpretability even when $R^2$ stays high, because SFD always includes dust behind the star while the RR Lyrae map stops at the stellar distance, and because Gaia completeness drops sharply in the crowded low-latitude fields.

### What Makes SFD Work and Fail

SFD works because dust in thermal equilibrium re-emits absorbed starlight as a modified blackbody, and the integrated far-infrared intensity is proportional to the total column density of dust. The temperature correction stabilizes the column estimate against sightline-to-sightline dust temperature variation. However, several failure modes are known:

- **Complex plane sightlines**: When multiple dust components at different distances and temperatures pile up along a single line of sight, the single-temperature assumption breaks down and the inferred column can be biased.
- **CMB contamination at high latitude**: At $|b| \gtrsim 60°$ the SFD signal is very faint and can be contaminated by CMB fluctuations and zodiacal light residuals, leading to artificial structures in the high-latitude reddening map ([Peek & Graves 2010](https://doi.org/10.1088/0004-637X/719/1/415)).
- **Planck-based improvements**: The higher angular resolution and better-constrained dust temperature from Planck allow the dust column to be estimated more accurately in complex regions. Meisner & Finkbeiner (2015) produced an improved dust map by fitting two-component models to the Planck and IRAS/DIRBE data, reducing systematic offsets in the SFD calibration.

### Interpretation of the Latitude-Dependent $R^2$ Table

The latitude-binned $R^2$ table should be interpreted empirically rather than from a fixed prior expectation. In the current notebook output, $R^2$ is actually strongest in the lowest-latitude bins: **$R^2 = {r2_0_10:.3f}$ for $0$–$10^\circ$** and **$R^2 = {r2_10_20:.3f}$ for $10$–$20^\circ$**, then it declines through the intermediate bins such as **$R^2 = {r2_20_30:.3f}$ for $20$–$30^\circ$** and **$R^2 = {r2_30_40:.3f}$ for $30$–$40^\circ$**. That does not mean the plane comparison is cleaner in an absolute sense. It means the plane has the largest dynamic range in dust, so $R^2$ can remain very high even though the two maps are not tracing the same total column to the same distance. High-latitude bins have a much smaller reddening range, so even modest noise or depth mismatch can reduce $R^2$ there.

The right physical conclusion is therefore two-part. First, the broad monotonic ordering of dusty versus clean sightlines agrees well enough that the RR Lyrae map is clearly recovering the same large-scale Galactic dust structure as SFD. Second, one should **not** expect exact map equality, especially in the plane, because the two products encode different observables: SFD is a 2D total-column map, while the RR Lyrae map is a distance-limited stellar reddening measurement.

### References

- Green, G. M., Schlafly, E. F., Zucker, C., et al. (2019), *A 3D Dust Map Based on Gaia, Pan-STARRS 1, and 2MASS*, ApJ, 887, 93. https://doi.org/10.3847/1538-4357/ab5362
- Meisner, A. M., & Finkbeiner, D. P. (2015), *The two-component dust spectral energy distribution of Planck and IRAS*, ApJ, 798, 88. https://doi.org/10.1088/0004-637X/798/2/88
- Peek, J. E. G., & Graves, G. J. (2010), *A CMB contamination analysis of the Schlegel et al. Galactic extinction map*, ApJ, 719, 415. https://doi.org/10.1088/0004-637X/719/1/415
- Schlegel, D. J., Finkbeiner, D. P., & Davis, M. (1998), *Maps of Dust Infrared Emission for Use in Estimation of Reddening and Cosmic Microwave Background Radiation Foregrounds*, ApJ, 500, 525. https://doi.org/10.1086/305772
- Schlafly, E. F., & Finkbeiner, D. P. (2011), *Measuring Reddening with Sloan Digital Sky Survey Stellar Spectra and Recalibrating SFD*, ApJ, 737, 103. https://doi.org/10.1088/0004-637X/737/2/103
"""

display(Markdown(discussion))

In [ ]:
from ugdatalab import plot_aitoff_reddening_dark

all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
dark_vmax = float(np.nanpercentile(
    np.asarray(rrlyrae_clean['E_bprp'], dtype=float), 99.5
))

fig_dark, ax_dark = plot_aitoff_reddening_dark(
    rrlyrae_clean,
    all_mask,
    vmin=0.0,
    vmax=dark_vmax,
    save=True,
)
plt.show()